---

### Del Tables

In [1]:
# #Delete all tables
# from pyspark.sql import SparkSession

# # Initialize the Spark session
# spark = SparkSession.builder.getOrCreate()

# # List of all tables in the Lakehouse
# tables = spark.catalog.listTables("UAT_DIEP2_EMBARK")  # Replace "Lakehouse" with your Lakehouse name

# # Loop through all tables and drop them
# for table in tables:
#     table_name = table.name
#     full_table_name = f"{table.database}.{table_name}" if table.database else table_name
#     spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
#     print(f"Deleted table: {full_table_name}")

StatementMeta(, 0f3c1162-0bd5-434c-90ae-6251e156c719, 3, Finished, Available, Finished)

In [2]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 3, Finished, Available, Finished)

### Initial Setup

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import json
from datetime import datetime 
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
from pyspark.sql.functions import col, lit, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType , DateType , BooleanType , DoubleType ,TimestampType,ArrayType,ArrayType,LongType
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType
spark = SparkSession.builder.appName("json-to-parquet").config("spark.driver.maxResultSize", "-1").getOrCreate()
from pyspark.sql.utils import AnalysisException
from delta.tables import *
import pandas as pd
import numpy as np
from pyspark.sql.functions import col, lit
import time

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 4, Finished, Available, Finished)

In [4]:
year=datetime.now().year
month = datetime.now().strftime('%m')

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, -1, Cancelled, , Cancelled)

In [5]:
client='EMBARK'
lob='HO5'
container='versions'

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 6, Finished, Available, Finished)

VACUUM only whichever container only as there will be clash if all table are considered

In [4]:
#add this in the start of each stageingzone notebook
#we create a table vacuummetadata for timestamp and clear old tombstone files


from pyspark.sql import SparkSession
from pyspark.sql import Row
from datetime import datetime
from delta.tables import DeltaTable

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Metadata table path and name
metadata_table_path = f"Tables/VacuumMetadata_{container}"


# Check if metadata table exists
if not DeltaTable.isDeltaTable(spark, metadata_table_path):
    # If not, create it with the initial timestamp (use an older date to ensure the first run)
    initial_timestamp = datetime.strptime("2023-01-01", "%Y-%m-%d")  # Replace with desired date
    spark.createDataFrame([Row(prev_timestamp=initial_timestamp.strftime("%Y-%m-%d"))]) \
         .write.format("delta").save(metadata_table_path)
else:
    print("Metadata table already exists.")



# Read the metadata table to get the previous timestamp
metadata_df = spark.read.format("delta").load(metadata_table_path)
prev_timestamp_str = metadata_df.select("prev_timestamp").collect()[0][0]
prev_timestamp = datetime.strptime(prev_timestamp_str, "%Y-%m-%d")

# Get current timestamp
current_timestamp = datetime.now()
days_difference = (current_timestamp - prev_timestamp).days

# Check if 7 days have passed
if days_difference >= 7:
    print("Running vacuum operation as timestamp difference is 7 days or more.")

    # List tables and run vacuum on each
    tables = spark.catalog.listTables("PROD_DIEP2_EMBARK")
    for table in tables:
        table_name = table.name
        if table_name.endswith(container):
            print(f"Vacuuming table: {table_name}")
            spark.sql(f"VACUUM {table_name} RETAIN 168 HOURS").show(truncate=False)

    # Update metadata table with the current timestamp
    new_timestamp = current_timestamp.strftime("%Y-%m-%d")
    new_metadata_df = spark.createDataFrame([Row(prev_timestamp=new_timestamp)])

    # Overwrite the metadata table with the new timestamp
    new_metadata_df.write.format("delta").mode("overwrite").save(metadata_table_path)
    print("Metadata timestamp updated.")
else:
    print("Vacuum operation skipped as timestamp difference is less than 7 days.")



StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 7, Finished, Available, Finished)

Metadata table already exists.


Vacuum operation skipped as timestamp difference is less than 7 days.


In [6]:
def policy_ref(data):
    id = data['id']
    return str(id)

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 8, Finished, Available, Finished)

### Fabric Integration

In [7]:
import json

# File path in your Lakehouse
file_path = f"/lakehouse/default/Files/LandingZone/{client}/{lob}/{container}/{client}_{lob}.json"

# Open the file and read it
with open(file_path, 'r',encoding='utf-8-sig') as f:
    landing_data = f.read()

# Split the file contents into separate lines if it's a multi-line JSON file
# landing_data_list = landing_data.strip().split('\n')

landing_data_list=json.loads(landing_data)

# Print length to verify that the data is loaded
print(len(landing_data_list))

landing_data=landing_data_list
len(landing_data)
storage_location="Tables"

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 9, Finished, Available, Finished)

95


### Merge Functionality

In [8]:
# def replace_nan_and_empty_list(value):
#     if isinstance(value, list) and len(value) == 0:
#         return None
#     elif isinstance(value, (np.ndarray, pd.Series)):
#         if pd.isna(value).all():
#             return None
#         else:
#             return value
#     elif isinstance(value, (float, int, str, type(None))):  # scalar types
#         if pd.isna(value):
#             return None
#         else:
#             return value
#     else:
#         return value

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 10, Finished, Available, Finished)

In [9]:
# import pandas as pd

# def merge_dataframes_fillna(df1, df2, on_columns, how='outer'):
#     """
#     Merges two DataFrames on the specified columns, fills missing values
#     in overlapping columns, and removes redundant columns with suffixes.

#     Parameters:
#     df1 (pd.DataFrame): The first DataFrame.
#     df2 (pd.DataFrame): The second DataFrame.
#     on_columns (list): The column names to merge on.
#     how (str): Merge method ('outer', 'inner', etc.).

#     Returns:
#     pd.DataFrame: The merged DataFrame with missing values handled.
#     """
    
#     # Merge the DataFrames on the given list of columns
#     merged_data = pd.merge(df1, df2, on=on_columns, how=how, suffixes=('_x', '_y'))

#     # Identify all columns except the merge key columns
#     common_columns = set(merged_data.columns) - set(on_columns)

#     # Loop through and fill missing values using the '_y' column first, then '_x'
#     for col in common_columns:
#         if col.endswith('_x'):
#             col_name = col[:-2]  # Remove '_x' to get the base column name
#             merged_data[col_name] = merged_data[f'{col_name}_y'].fillna(merged_data[f'{col_name}_x'])

#     # Drop the extra columns with '_x' and '_y' suffixes
#     merged_data = merged_data.drop(columns=[col for col in merged_data.columns if col.endswith(('_x', '_y'))])

#     return merged_data



# from pyspark.sql.functions import col, lit
# from delta.tables import DeltaTable
# from functools import reduce
# import operator
# import traceback

# def handle_missing_columns(existing_df, data):
#     """
#     Helper function to add missing columns with proper null handling.
#     """
#     for column in existing_df.columns:
#         if column not in data.columns:
#             # Add missing column with appropriate type handling
#             data[column] = None  # If you need it as string, you can adjust it like below
#             # data[column] = data[column].astype(str)

#         # If the entire column in data is null, cast it to string type for consistency
#         # if data[column].isna().sum() == data.shape[0]:
#         #     data[column] = data[column].astype(str)
    
#     return data

# def mergeandinsert(table_name, data, key_columns):
#     delta_table_path = f"{storage_location}/{table_name}_{container}"

#     if data.empty==True:
#         print("Skip since empty")
#         return
    
#     columnss=[]
#     for i in data.columns:
#         if data[i].isna().sum()==data.shape[0]:
#             columnss.append(i)
#     colus=set(data.columns)-set(columnss)
#     data=data.loc[:,list(colus)]        
#         #     data[i]=data[i].astype(str)

#     data=data.replace({np.nan:None,'NaN':None,'nan':None,np.NaN:None})    

#     data = data.map(replace_nan_and_empty_list)

#     try:
#         delta_table = DeltaTable.forPath(spark, delta_table_path)
#     except AnalysisException:
#         delta_table = None
    
#     if delta_table is None:
#         dfs = spark.createDataFrame(data)
#         dfs = dfs.na.replace(float('nan'), None)
#         dfs = dfs.na.replace(float('NaN'), None)

#         dfs.write.format('delta').save(delta_table_path)
#     else:
#         try:
#             existing_df = spark.read.format("delta").load(delta_table_path).toPandas()

#             data = handle_missing_columns(existing_df, data)
#             schema = delta_table.toDF().schema
#             dfs = spark.createDataFrame(data,schema)
#             dfs = dfs.na.replace(float('nan'), None)
#             dfs = dfs.na.replace(float('NaN'), None)

#             merge_condition = reduce(operator.and_, [col(f"df.{col_name}") == col(f"old_data.{col_name}") for col_name in key_columns])

#             delta_table.alias("old_data").merge(dfs.alias("df"), merge_condition) \
#                 .whenMatchedUpdateAll() \
#                 .whenNotMatchedInsertAll() \
#                 .execute()

#         except Exception as e:
#             print(f"Exception occurred: {e}")
#             print(traceback.format_exc())  # Print the full stack trace for better debugging
#             existing_df = spark.read.format("delta").load(delta_table_path).toPandas()

#             try:
#                 data = handle_missing_columns(existing_df, data)
#                 data1=merge_dataframes_fillna(existing_df,data,key_columns)
#                 existing_dfs = spark.createDataFrame(data1)
#                 existing_dfs = existing_dfs.na.replace(float('nan'), None)
#                 existing_dfs = existing_dfs.na.replace(float('NaN'), None)
#                 existing_dfs.write.format("delta").mode("overwrite").option("overwriteSchema", "True").save(delta_table_path)
            
#             except:
#                 data = handle_missing_columns(existing_df, data)
#                 existing_df = handle_missing_columns(data, existing_df)

#                 existing_dfs = spark.createDataFrame(existing_df)
#                 existing_dfs = existing_dfs.na.replace(float('nan'), None)
#                 existing_dfs = existing_dfs.na.replace(float('NaN'), None)
#                 existing_dfs.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_table_path)
                
#                 delta_table = DeltaTable.forPath(spark, delta_table_path)
#                 dfs = spark.createDataFrame(data)

#                 merge_condition = reduce(operator.and_, [col(f"df.{col_name}") == col(f"old_data.{col_name}") for col_name in key_columns])
                
#                 delta_table.alias("old_data").merge(dfs.alias("df"), merge_condition) \
#                     .whenMatchedUpdateAll() \
#                     .whenNotMatchedInsertAll() \
#                     .execute()


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 11, Finished, Available, Finished)

In [10]:
# import traceback
# from functools import reduce
# from pyspark.sql.functions import col, lit, struct
# from delta.tables import DeltaTable
# from pyspark.sql.utils import AnalysisException
# import pandas as pd


# def add_missing_top_level_columns(base_df, target_df):
#     """
#     Ensure target_df has all top-level columns present in base_df by adding them as nulls.
#     Only handles top-level columns (keeps structs intact).
#     """
#     base_cols = set(base_df.columns)
#     target_cols = set(target_df.columns)

#     # Add missing in target
#     for c in base_cols - target_cols:
#         target_df = target_df.withColumn(c, lit(None))
#     # Add missing in base
#     for c in target_cols - base_cols:
#         base_df = base_df.withColumn(c, lit(None))

#     target_df = target_df.select(base_df.columns)  # reorder
#     return base_df, target_df


# # Helper: build a struct Column expression out of available flat columns
# def _build_struct_expr_and_used_cols(df, parent_name, struct_schema):
#     """
#     Build a struct(...) Column for parent_name from columns like:
#       - "parent_name.sub"   (literal dotted column)
#       - "parent_name_sub"    (underscore flattened)
#       - "sub"                (top-level, fallback; not dropped)
#     Returns (ColumnExpression | None, list_of_used_column_names_to_drop)
#     """
#     exprs = []
#     used = []

#     for sub in struct_schema.fields:
#         full = f"{parent_name}.{sub.name}"            # dotted form
#         undersc = f"{parent_name}_{sub.name}"         # underscore form
#         # If nested struct -> recursive
#         if isinstance(sub.dataType, StructType):
#             nested_expr, nested_used = _build_struct_expr_and_used_cols(df, full, sub.dataType)
#             # also try underscore-prefixed nested recursively
#             if nested_expr is None and undersc in df.columns:
#                 # try treat undersc as a struct source: but usually underscore flattened will be leaf columns,
#                 # so we attempt to find deeper underscore children dynamically by scanning df.columns with prefix.
#                 # For simplicity, attempt recursive on prefix with dot replaced:
#                 nested_expr, nested_used = _build_struct_expr_and_used_cols(df, undersc, sub.dataType)
#             if nested_expr is not None:
#                 exprs.append(nested_expr.alias(sub.name))
#                 used.extend(nested_used)
#             else:
#                 # no children found -> put null for this subfield
#                 exprs.append(lit(None).alias(sub.name))
#             continue

#         # Non-struct leaf: prefer dotted literal, then underscore, then top-level column
#         if full in df.columns:
#             # use backticks to reference literal column names that include dots
#             exprs.append(col(f"`{full}`").alias(sub.name))
#             used.append(full)
#         elif undersc in df.columns:
#             exprs.append(col(undersc).alias(sub.name))
#             used.append(undersc)
#         elif sub.name in df.columns:
#             # fallback to top-level column (do NOT add to used list; we won't drop it)
#             exprs.append(col(sub.name).alias(sub.name))
#         else:
#             # missing -> use null
#             exprs.append(lit(None).alias(sub.name))

#     if not exprs:
#         return None, []
#     return struct(*exprs), used


# def rebuild_structs(df, target_schema):
#     """
#     Rebuild StructType columns in `df` to match target_schema.
#     - Looks for flattened columns like "Parent.Child" (dotted) or "Parent_Child" (underscore)
#       and rebuilds `Parent` as a struct.
#     - Drops only the flattened literal columns that were used.
#     """
#     # Track all columns dropped in this call to avoid repeated drops
#     cols_to_drop = []

#     for field in target_schema.fields:
#         if not isinstance(field.dataType, StructType):
#             continue  # only rebuild struct fields

#         # if df already has a proper struct column, skip
#         if field.name in df.columns and isinstance(df.schema[field.name].dataType, StructType):
#             # We may still want to recursively fix deeper nested structs if necessary,
#             # but if it's already StructType, assume schema matches or is compatible.
#             continue

#         # Build struct expression (if any) using available flattened columns
#         struct_expr, used_cols = _build_struct_expr_and_used_cols(df, field.name, field.dataType)

#         if struct_expr is not None:
#             # create/replace the struct column
#             df = df.withColumn(field.name, struct_expr)
#             # collect columns we should drop (only dotted/underscore ones)
#             cols_to_drop.extend([c for c in used_cols if c in df.columns])

#     # Now drop all used flattened columns (if present)
#     if cols_to_drop:
#         # dedupe
#         cols_to_drop = list(dict.fromkeys(cols_to_drop))
#         df = df.drop(*cols_to_drop)

#     return df


# def build_merge_condition(key_columns, left_alias="old", right_alias="new"):
#     if not key_columns:
#         raise ValueError("key_columns must be provided")
#     return reduce(lambda a, b: a & b, [col(f"{left_alias}.{k}") == col(f"{right_alias}.{k}") for k in key_columns])


# def mergeandinsert(table_name, data, key_columns, retry_on_schema_change=True):
#     delta_table_path = f"{storage_location}/{table_name}_{container}"

#     # Convert pandas → Spark
#     if isinstance(data, pd.DataFrame):
#         if data.empty:
#             print("Skip since empty")
#             return
#         dfs = spark.createDataFrame(data)
#     else:
#         dfs = data

#     # Drop all-null columns if pandas
#     try:
#         if isinstance(data, pd.DataFrame):
#             all_null_cols = [c for c in data.columns if data[c].isna().sum() == data.shape[0]]
#             if all_null_cols:
#                 dfs = dfs.drop(*all_null_cols)
#     except Exception:
#         pass

#     # Try to get existing Delta table
#     try:
#         delta_table = DeltaTable.forPath(spark, delta_table_path)
#         table_exists = True
#     except Exception:
#         delta_table = None
#         table_exists = False

#     if not table_exists:
#         dfs.write.format("delta").save(delta_table_path)
#         print(f"Created new Delta table at {delta_table_path}")
#         return

#     # Table exists
#     attempts = 0
#     max_attempts = 2 if retry_on_schema_change else 1
#     last_exc = None

#     while attempts < max_attempts:
#         attempts += 1
#         try:
#             existing_df = delta_table.toDF()
#             existing_schema = existing_df.schema

#             # Rebuild structs in new DF to match schema
#             dfs_aligned = rebuild_structs(dfs, existing_schema)

#             # Align top-level columns
#             existing_df, dfs_aligned = add_missing_top_level_columns(existing_df, dfs_aligned)

#             # Merge
#             merge_condition = build_merge_condition(key_columns)
#             delta_table.alias("old").merge(
#                 dfs_aligned.alias("new"),
#                 merge_condition
#             ).whenMatchedUpdateAll() \
#              .whenNotMatchedInsertAll() \
#              .execute()

#             print(f"Merged into Delta table at {delta_table_path}")
#             return

#         except AnalysisException as ae:
#             last_exc = ae
#             msg = str(ae)
#             print(f"Delta merge AnalysisException (attempt {attempts}): {msg}")
#             if "DELTA_SCHEMA_CHANGE_SINCE_ANALYSIS" in msg and attempts < max_attempts:
#                 print("Schema changed since analysis, retrying...")
#                 continue
#             break
#         except Exception as e:
#             last_exc = e
#             print(f"Delta merge failed: {e}")
#             break

#     # Fallback: union + dedupe
#     try:
#         existing_df = delta_table.toDF()
#         existing_df, dfs_aligned = add_missing_top_level_columns(existing_df, dfs)
#         combined = dfs_aligned.unionByName(existing_df).dropDuplicates(key_columns)
#         combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_table_path)
#         print(f"Fallback overwrite completed at {delta_table_path}")
#     except Exception as final_e:
#         print("Final fallback failed.")
#         raise last_exc if last_exc else final_e


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 12, Finished, Available, Finished)

In [11]:
# spark.conf.set("spark.driver.maxResultSize", "8g")

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 13, Finished, Available, Finished)

In [8]:
# ### Code in Pandas but facing memory issue due to large size for transforming table to pandas.
# import pandas as pd
# import numpy as np
# from pyspark.sql.functions import col
# from delta.tables import DeltaTable
# from functools import reduce
# import operator
# import traceback
# from pyspark.sql.utils import AnalysisException

# def replace_nan_and_empty_list(value):
#     if isinstance(value, list) and len(value) == 0:
#         return None
#     elif isinstance(value, (np.ndarray, pd.Series)):
#         if pd.isna(value).all():
#             return None
#         else:
#             return value
#     elif isinstance(value, (float, int, str, type(None))):  # scalar types
#         if pd.isna(value):
#             return None
#         else:
#             return value
#     else:
#         return value

# def merge_dataframes_fillna(df1, df2, on_columns, how='outer'):
#     print("Starting Pandas merge with fillna...")
#     merged_data = pd.merge(df1, df2, on=on_columns, how=how, suffixes=('_x', '_y'))

#     common_columns = set(merged_data.columns) - set(on_columns)

#     for col_name in common_columns:
#         if col_name.endswith('_x'):
#             base_name = col_name[:-2]
#             merged_data[base_name] = merged_data[f'{base_name}_y'].fillna(merged_data[f'{base_name}_x'])
#             print(f"Filled missing values for column: {base_name}")

#     merged_data = merged_data.drop(columns=[col for col in merged_data.columns if col.endswith(('_x', '_y'))])
#     print("Pandas merge complete.")
#     return merged_data

# def handle_missing_columns(existing_df, data):
#     print("Checking for missing columns...")
#     for column in existing_df.columns:
#         if column not in data.columns:
#             print(f"Adding missing column: {column}")
#             data[column] = None
#     return data

# def mergeandinsert(table_name, data, key_columns):
#     delta_table_path = f"{storage_location}/{table_name}_{container}"
#     print(f"Delta table path: {delta_table_path}")

#     if data.empty:
#         print("DataFrame is empty. Skipping merge.")
#         return

#     # Drop columns that are all null
#     columnss = [i for i in data.columns if data[i].isna().sum() == data.shape[0]]
#     colus = set(data.columns) - set(columnss)
#     data = data.loc[:, list(colus)]
#     print(f"Columns after removing fully null columns: {data.columns.tolist()}")

#     # Replace NaNs
#     data = data.replace({np.nan: None, 'NaN': None, 'nan': None, np.NaN: None})
#     print("Replaced NaNs with None in DataFrame.")

#     # Your custom map function if exists
#     data = data.map(replace_nan_and_empty_list)
#     print("Applied replace_nan_and_empty_list function.")

#     try:
#         delta_table = DeltaTable.forPath(spark, delta_table_path)
#         print("Delta table found.")
#     except AnalysisException:
#         delta_table = None
#         print("Delta table does not exist. Will create new table.")

#     if delta_table is None:
#         print("Creating new Delta table...")
#         dfs = spark.createDataFrame(data)
#         dfs.write.format('delta').save(delta_table_path)
#         print("New Delta table created successfully.")
#     else:
#         try:
#             print("Attempting to read existing Delta table to Pandas...")
#             existing_df = spark.read.format("delta").load(delta_table_path).toPandas()
#             print(f"Existing Delta table loaded. Shape: {existing_df.shape}")

#             data = handle_missing_columns(existing_df, data)
#             schema = delta_table.toDF().schema
#             dfs = spark.createDataFrame(data, schema)
#             print("Data converted to Spark DataFrame with matching schema.")

#             merge_condition = reduce(
#                 operator.and_,
#                 [col(f"df.{c}") == col(f"old_data.{c}") for c in key_columns]
#             )
#             print(f"Merge condition created for keys: {key_columns}")

#             delta_table.alias("old_data").merge(
#                 dfs.alias("df"), merge_condition
#             ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
#             print("Delta merge executed successfully.")

#         except Exception as e:
#             print(f"Exception during Delta merge: {e}")
#             print(traceback.format_exc())

#             print("Falling back to Pandas merge...")
#             existing_df = spark.read.format("delta").load(delta_table_path).toPandas()
#             data = handle_missing_columns(existing_df, data)
#             data1 = merge_dataframes_fillna(existing_df, data, key_columns)

#             existing_dfs = spark.createDataFrame(data1)
#             existing_dfs.write.format("delta").mode("overwrite").option("overwriteSchema", "True").save(delta_table_path)
#             print("Fallback Pandas merge written back to Delta table successfully.")


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 14, Finished, Available, Finished)

In [13]:
# ## In pyspark facing issue with columns named with "dot"
# from pyspark.sql import DataFrame
# from pyspark.sql.functions import col, lit, when, size, array
# from pyspark.sql.types import StringType
# from delta.tables import DeltaTable
# from functools import reduce
# import operator
# import traceback
# import numpy as np

# # --------------------------
# # UDF to replace NaNs / empty lists
# # --------------------------
# def replace_nan_and_empty_list_py(value):
#     if isinstance(value, list) and len(value) == 0:
#         return None
#     elif isinstance(value, (np.ndarray,)):
#         if np.isnan(value).all():
#             return None
#         else:
#             return value.tolist()
#     elif isinstance(value, (float, int, str, type(None))):
#         if value is None or (isinstance(value, float) and np.isnan(value)):
#             return None
#         else:
#             return value
#     else:
#         return value

# from pyspark.sql.functions import udf
# replace_nan_and_empty_list_udf = udf(replace_nan_and_empty_list_py, StringType())

# # --------------------------
# # Handle missing columns
# # --------------------------
# def handle_missing_columns(existing_df: DataFrame, new_df: DataFrame) -> DataFrame:
#     print("Checking for missing columns...")
#     for column in existing_df.columns:
#         if column not in new_df.columns:
#             print(f"Adding missing column: {column}")
#             new_df = new_df.withColumn(column, lit(None))
#     return new_df

# # --------------------------
# # Fill missing values for overlapping columns (_x/_y style) in Spark
# # --------------------------
# def fill_missing_columns(existing_df: DataFrame, new_df: DataFrame, key_columns: list) -> DataFrame:
#     for col_name in existing_df.columns:
#         if col_name not in key_columns:
#             if col_name in new_df.columns:
#                 # Fill nulls in new_df from existing_df
#                 new_df = new_df.withColumn(f"`{col_name}`",
#                            when(col(f"`{col_name}`").isNotNull(), col(f"`{col_name}`"))
#                            .otherwise(lit(None)))

#             else:
#                 new_df = new_df.withColumn(col_name, lit(None))
#     return new_df

# # --------------------------
# # Merge & insert to Delta (PySpark only)
# # --------------------------
# def mergeandinsert(table_name: str, data, key_columns: list):
#     # Convert Pandas to Spark DataFrame if needed
#     if isinstance(data, pd.DataFrame):
#         print("Converting Pandas DataFrame to Spark DataFrame...")
#         data = spark.createDataFrame(data)

#     delta_table_path = f"{storage_location}/{table_name}_{container}"
#     print(f"Delta table path: {delta_table_path}")

#     if data.rdd.isEmpty():
#         print("DataFrame is empty. Skipping merge.")
#         return

#     # Remove fully null columns safely
#     non_null_cols = [c for c in data.columns if data.filter(col(f"`{c}`").isNotNull()).count() > 0]
#     data = data.select(*[f"`{c}`" for c in non_null_cols])
#     print(f"Columns after removing fully null columns: {non_null_cols}")

#     # Apply replace_nan_and_empty_list UDF to all columns
#     for c in data.columns:
#         data = data.withColumn(f"`{c}`", replace_nan_and_empty_list_udf(col(f"`{c}`")))
#     print("Applied replace_nan_and_empty_list function to all columns.")

#     # Build rename map for columns with '.' but without '_'
#     rename_map = {c: c.replace('.', '_') for c in data.columns if '.' in c and '_' not in c}

#     # Rename columns in data for Delta merge
#     data = data.toDF(*[rename_map.get(c, c) for c in data.columns])

#     # Access Delta table
#     try:
#         delta_table = DeltaTable.forPath(spark, delta_table_path)
#         print("Delta table found.")
#     except Exception:
#         delta_table = None
#         print("Delta table does not exist. Will create new table.")

#     if delta_table is None:
#         print("Creating new Delta table...")
#         data.write.format("delta").save(delta_table_path)
#         print("New Delta table created successfully.")
#     else:
#         try:
#             existing_df = delta_table.toDF()
#             print(f"Existing Delta table loaded. Columns: {existing_df.columns}")

#             # Rename columns in existing_df to match rename_map
#             existing_df = existing_df.toDF(*[rename_map.get(c, c) for c in existing_df.columns])

#             # Handle missing columns
#             data = handle_missing_columns(existing_df, data)

#             # Merge condition using backticks for all key columns
#             merge_condition = reduce(
#                 operator.and_,
#                 [col(f"`{c}`") == col(f"old_data.`{c}`") for c in key_columns]
#             )

#             print(f"Merge condition created for keys: {key_columns}")

#             # Perform merge
#             delta_table.alias("old_data") \
#                 .merge(data.alias("df"), merge_condition) \
#                 .whenMatchedUpdateAll() \
#                 .whenNotMatchedInsertAll() \
#                 .execute()
#             print("Delta merge executed successfully.")

#             # Optional: rename columns back in data for readability
#             for original, new in rename_map.items():
#                 data = data.withColumnRenamed(new, original)

#         except Exception as e:
#             print(f"Exception during Delta merge: {e}")
#             print(traceback.format_exc())
#             print("Falling back to union & overwrite...")

#             existing_df = delta_table.toDF()
#             data = handle_missing_columns(existing_df, data)

#             # Fill missing overlapping columns
#             data = fill_missing_columns(existing_df, data, key_columns)

#             # Union the DataFrames
#             merged_df = existing_df.unionByName(data)
#             merged_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_table_path)
#             print("Fallback merge completed using union and overwrite.")


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 15, Finished, Available, Finished)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import NullType, StructType, StringType, ArrayType
from delta.tables import DeltaTable
from pyspark.sql.functions import when, col, isnan
from pyspark.sql.types import (
    StringType, BooleanType, IntegerType, LongType,
    DoubleType, FloatType, DecimalType, DateType, TimestampType
)

def mergeandinsert(table_name, data, key_columns):
    # Escape dotted column names
    def esc(col_name):
        return f"`{col_name}`"
    """
    Merge new data into a Delta table, handling dotted column names safely,
    flattening structs, handling arrays of structs, and avoiding driver memory overflow.
    
    Parameters:
    - table_name: str, name of the table
    - data: Pandas or Spark DataFrame with new data
    - key_columns: list of str, business keys (may contain dots)
    """

    delta_table_path = f"{storage_location}/{table_name}_{container}"

    # Convert Pandas to Spark if needed
    if isinstance(data, pd.DataFrame):
        if data.empty:
            print("Skip since empty")
            return
        data_spark = spark.createDataFrame(data)
    else:
        data_spark = data
        if data_spark.rdd.isEmpty():
            print("Skip since empty")
            return

    # Replace problematic NaN/inf values in Spark
    # data_spark = data_spark.replace(float("nan"), None).replace(float("inf"), None)
    # Convert NaN/Inf to NULL properly
    for field in data_spark.schema.fields:
        if isinstance(field.dataType, (DoubleType, FloatType)):
            data_spark = data_spark.withColumn(
                field.name,
                when(isnan(col(esc(field.name))) | (col(esc(field.name)) == float("inf")), None).otherwise(col(esc(field.name)))
            )

    # --- Helper: flatten structs safely ---
    def safe_flatten(df):
        while True:
            struct_fields = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
            if not struct_fields:
                break
            for s in struct_fields:
                nested_fields = df.schema[s].dataType.fields
                for n in nested_fields:
                    df = df.withColumn(f"{s}.{n.name}", F.col(s).getField(n.name))
                df = df.drop(s)
        return df

    # --- Helper: explode array-of-structs columns ---
    def explode_arrays(df):
        array_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, ArrayType) and isinstance(f.dataType.elementType, StructType)]
        for col_name in array_cols:
            df = df.withColumn(col_name, F.explode_outer(F.col(col_name)))
        return df

    # Apply safe flatten and array explode
    data_spark = explode_arrays(safe_flatten(data_spark))

    for field in data_spark.schema.fields:
        if isinstance(field.dataType, NullType):
            col_name_esc = f"`{field.name}`"  # escape dotted column
            data_spark = data_spark.withColumn(field.name, F.col(col_name_esc).cast(StringType()))

    # # Cast NullType columns to string to avoid merge errors
    # for field in data_spark.schema.fields:
    #     if isinstance(field.dataType, NullType):
    #         data_spark = data_spark.withColumn(field.name, F.col(field.name).cast(StringType()))

    

    # Try to load Delta table
    try:
        delta_table = DeltaTable.forPath(spark, delta_table_path)
    except:
        delta_table = None

    if delta_table is None:
        # First time write
        data_spark.write.format("delta").mode("overwrite").save(delta_table_path)
        print(f"Created new Delta table at {delta_table_path}")
    else:
        spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
        # Enable auto schema merge
        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
            

        # Align schema types
        delta_schema = {f.name: f.dataType for f in delta_table.toDF().schema}
        for field in data_spark.schema.fields:
            if field.name in delta_schema:
                target_type = delta_schema[field.name]
                col_name_esc = f"`{field.name}`"
                if not isinstance(field.dataType, type(target_type)):
                    # --- Numeric types ---
                    if isinstance(target_type, (IntegerType, LongType)):
                        # Handle ints/longs, cleaning float-like strings ("1364.0")
                        data_spark = data_spark.withColumn(
                            field.name,
                            F.regexp_replace(F.col(col_name_esc).cast("string"), r"\.0$", "").cast("long")
                        )

                    elif isinstance(target_type, (FloatType, DoubleType, DecimalType)):
                        # Cast anything to double/decimal safely
                        data_spark = data_spark.withColumn(
                            field.name,
                            F.col(col_name_esc).cast("double")
                        )

                    # --- Boolean type ---
                    elif isinstance(target_type, BooleanType):
                        # Handle strings like "true"/"false", numbers like 0/1
                        data_spark = data_spark.withColumn(
                            field.name,
                            F.when(F.col(col_name_esc).isin("true", "True", "1"), F.lit(True))
                            .when(F.col(col_name_esc).isin("false", "False", "0"), F.lit(False))
                            .otherwise(None)
                            .cast("boolean")
                        )

                    # --- Date type ---
                    elif isinstance(target_type, DateType):
                        # Expect yyyy-MM-dd or convertable formats
                        data_spark = data_spark.withColumn(
                            field.name,
                            F.to_date(F.col(col_name_esc).cast("string"), "yyyy-MM-dd")
                        )

                    # --- Timestamp type ---
                    elif isinstance(target_type, TimestampType):
                        data_spark = data_spark.withColumn(
                            field.name,
                            F.to_timestamp(F.col(col_name_esc).cast("string"))
                        )

                    # --- String type or fallback ---
                    else:
                        # Default: cast everything else to string
                        data_spark = data_spark.withColumn(
                            field.name,
                            F.col(col_name_esc).cast("string")
                        )

                # if not isinstance(field.dataType, type(target_type)):
                #     # Safer: always cast to string instead of target_type
                #     data_spark = data_spark.withColumn(field.name, F.col(col_name_esc).cast(StringType()))

                # if not isinstance(field.dataType, type(target_type)):
                #     if isinstance(target_type, NullType):
                #         data_spark = data_spark.withColumn(col_name_esc, F.col(col_name_esc).cast(StringType()))
                #     else:
                #         data_spark = data_spark.withColumn(col_name_esc, F.col(col_name_esc).cast(target_type))


        # Build merge condition using backticks
        merge_condition = " AND ".join([f"old_data.{esc(cols)} = df.{esc(cols)}" for cols in key_columns])
        print('Merge_condition = ',merge_condition)

        # Perform merge
        delta_table.alias("old_data") \
            .merge(data_spark.alias("df"), merge_condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print(f"Merged new data into {delta_table_path}")


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 16, Finished, Available, Finished)

### Data Issue finding block

In [14]:
# import pandas as pd
# import numpy as np

# df_missing_unitid = []

# for jsonn in landing_data:
#     ref = policy_ref(jsonn)
#     if jsonn['Risks']['Properties']:
#         for data in jsonn['Risks']['Properties']:
#             if "UnitId" not in data:
#                 df_missing_unitid.append({'Policy_ref': ref, **data})

# df_missing_unitid = pd.json_normalize(df_missing_unitid)
# df_missing_unitid.replace({np.nan: None}, inplace=True)

# # Show policies missing UnitId
# print("Policies missing UnitId:")
# print(df_missing_unitid)


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 176, Finished, Available, Finished)

### PolicyStatusHistory

In [9]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    policystatushistory_id = 0
    if i['PolicyStatusHistory']:
        for j in i['PolicyStatusHistory']:
            policystatushistory_id += 1
            j = {'Policy_ref': ref, 'PolicyStatusHistory_Id': policystatushistory_id, **j}
            data_list.append(j)
data = pd.json_normalize(data_list)
data

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 177, Finished, Available, Finished)

,Policy_ref,PolicyStatusHistory_Id,Id,OldStatus,NewStatus,Remarks,ChangedBy,ChangedDate
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,NaN,,Quote Indication,,Meritage Home Insurance Agency,2025-09-22T13:29:45.984Z
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2,NaN,Quote Indication,Submission,,Meritage Home Insurance Agency,2025-09-22T13:29:49.091Z
2,111b5c8b-30b3-427f-977b-fe5ccd358b5f,3,NaN,Submission,Quote Offered,,Meritage Home Insurance Agency,2025-09-22T13:31:35.441Z
3,111b5c8b-30b3-427f-977b-fe5ccd358b5f,4,NaN,Policy In Force,Policy In Force,,Abigail Miller,2025-09-22T13:38:29.251Z
4,ae46254b-2338-44ed-8238-486145c927fc,1,NaN,,Quote Indication,,Underwriter User,2025-09-24T07:53:08.252Z
...,...,...,...,...,...,...,...,...
784,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,3,NaN,Submission,Submission,,Underwriter User,2025-10-15T12:52:05.466Z
785,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,4,NaN,Submission,Quote Offered,,Underwriter User,2025-10-15T12:52:16.814Z
786,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,5,NaN,Policy In Force,Policy In Force,None,Underwriter User,2025-10-15T12:52:28.646Z
787,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,6,NaN,Policy In Force,Endorsement Initiated,,Underwriter User,2025-10-15T12:55:37.39Z


In [10]:
mergeandinsert('PolicyStatusHistory',data,['Policy_ref','PolicyStatusHistory_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 178, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`PolicyStatusHistory_Id` = df.`PolicyStatusHistory_Id`


Merged new data into Tables/PolicyStatusHistory_versions


### DiscountAndSurcharges

In [17]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    DiscountAndSurcharges_id = 0
    try:
        j = i['DiscountAndSurcharges']
        DiscountAndSurcharges_id += 1
        j = {'Policy_ref': ref, 'DiscountAndSurcharges_Id': DiscountAndSurcharges_id, **j}
        data_list.append(j)
    except:
        DiscountAndSurcharges_id += 1
        j = {'Policy_ref': ref, 'DiscountAndSurcharges_Id': DiscountAndSurcharges_id}
        data_list.append(j)

data = pd.json_normalize(data_list)
data

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 179, Finished, Available, Finished)

,Policy_ref,DiscountAndSurcharges_Id,Description
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,
1,ae46254b-2338-44ed-8238-486145c927fc,1,
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,
...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,


In [18]:
mergeandinsert('DiscountAndSurcharges',data,['Policy_ref','DiscountAndSurcharges_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 180, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`DiscountAndSurcharges_Id` = df.`DiscountAndSurcharges_Id`


Merged new data into Tables/DiscountAndSurcharges_versions


### PolicyRiskAttributes

In [19]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    if i['PolicyRiskAttributes']:
        j = i['PolicyRiskAttributes']
        j = {'Policy_ref': ref, **j}
        data_list.append(j)

data = pd.json_normalize(data_list)


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 181, Finished, Available, Finished)

In [20]:
# Please review the columns

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 182, Finished, Available, Finished)

In [21]:
try:
       data=data.loc[:,['Policy_ref', 'FirstName', 'MiddleName', 'LastName',
       'EstimatedBalanceAfterCancel', 'BillingAccountNumber',
       'CurrentPolicyBalance', 'DropboxSignatureRequestID',
       'CancellationCount', 'ReinstatementCount', 'EndorsementCount',
       'InstallmentDueDate', 'CurrentInstallmentAmount', 'ExcludedDriverList','OriginalInceptionDate', 'IsNonRenewal'
       , 'NonRenewalDate', 'SISCurrentPayment',
       'SISRenewalTermDownPayment', 'IsNonRenewalDocGenerated']]
       # , 'NonRenewalTmp.IsNonRenewal',
       # 'NonRenewalTmp.NonRenewalReasons', 'NonRenewalTmp.NonRenewalRemarks',
       # 'NonRenewalTmp.NonRenewalDate','SweepLocation']]
except:
       data=data.reindex(columns=['Policy_ref', 'FirstName', 'MiddleName', 'LastName',
       'EstimatedBalanceAfterCancel', 'BillingAccountNumber',
       'CurrentPolicyBalance', 'DropboxSignatureRequestID',
       'CancellationCount', 'ReinstatementCount', 'EndorsementCount',
       'InstallmentDueDate', 'CurrentInstallmentAmount', 'ExcludedDriverList', 'OriginalInceptionDate',']IsNonRenewal'
       , 'NonRenewalDate', 'SISCurrentPayment',
       'SISRenewalTermDownPayment', 'IsNonRenewalDocGenerated']).fillna(' ')     
       # , 'NonRenewalTmp.IsNonRenewal',
       # 'NonRenewalTmp.NonRenewalReasons', 'NonRenewalTmp.NonRenewalRemarks',
       # 'NonRenewalTmp.NonRenewalDate','SweepLocation']]
         

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 183, Finished, Available, Finished)

In [22]:
mergeandinsert('PolicyRiskAttributes',data,['Policy_ref'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 184, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/PolicyRiskAttributes_versions


### Attributes

In [23]:
data_list = []
dl=[]
ds=[]
for i in landing_data:
    ref = policy_ref(i)
    if i['Attributes']:
        j = i['Attributes']
        if 'LobsArr' in j and j['LobsArr']:
            lid=0
            for k in j['LobsArr']:
                lid+=1
                datal={'Policy_ref':ref,'LobId':lid,**k}
                dl.append(datal)

        if 'StatesArr' in j and j['StatesArr']:
            sid=0
            for k in j['StatesArr']:
                sid+=1
                datas={'Policy_ref':ref,'States_Id':sid,**k}
                ds.append(datas)
        j = {'Policy_ref': ref, **j}
        data_list.append(j)
          

data = pd.json_normalize(data_list)
data_lob=pd.json_normalize(dl)
data_states=pd.json_normalize(ds)

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 185, Finished, Available, Finished)

In [24]:
data.columns

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 186, Finished, Available, Finished)

Index(['Policy_ref', 'Client', 'AppSource', 'Carrier', 'Coverholder', 'Lob',
       'State', 'Product', 'RaterVersion', 'IsMasterPolicy',
       'MasterReferenceNumber', 'IsTestPolicy', 'UISchemaVersion',
       'AgentEffectiveDate', 'NoOfClaims', 'IsCoverageSaved',
       'AdjustPremiumType', 'AdjustPremiumReason', 'AdjustPremiumOtherReason',
       'CanAgentApproveEnd', 'QuoteValidity.Value', 'QuoteValidity.ValueType',
       'RenewalConfigurations.AfterForUnderwriter.Value',
       'RenewalConfigurations.AfterForUnderwriter.ValueType',
       'RenewalConfigurations.BeforeForUnderwriter.Value',
       'RenewalConfigurations.BeforeForUnderwriter.ValueType',
       'RenewalConfigurations.AfterForAgent.Value',
       'RenewalConfigurations.AfterForAgent.ValueType',
       'RenewalConfigurations.BeforeForAgent.Value',
       'RenewalConfigurations.BeforeForAgent.ValueType', 'Subjectivity'],
      dtype='object')

In [25]:
try:
    data=data.loc[:['Policy_ref', 'Client', 'AppSource', 'Carrier', 'Coverholder', 'Lob',
       'State', 'Product', 'RaterVersion', 'IsMasterPolicy',
       'MasterReferenceNumber', 'IsTestPolicy', 'UISchemaVersion',
       'AgentEffectiveDate', 'NoOfClaims', 'IsCoverageSaved',
       'AdjustPremiumType', 'AdjustPremiumReason', 'AdjustPremiumOtherReason',
       'CanAgentApproveEnd', 'QuoteValidity.Value', 'QuoteValidity.ValueType',
       'RenewalConfigurations.AfterForUnderwriter.Value',
       'RenewalConfigurations.AfterForUnderwriter.ValueType',
       'RenewalConfigurations.BeforeForUnderwriter.Value',
       'RenewalConfigurations.BeforeForUnderwriter.ValueType',
       'RenewalConfigurations.AfterForAgent.Value',
       'RenewalConfigurations.AfterForAgent.ValueType',
       'RenewalConfigurations.BeforeForAgent.Value',
       'RenewalConfigurations.BeforeForAgent.ValueType', 'Subjectivity']]
except:
    data=data.reindex(columns=['Policy_ref', 'Client', 'AppSource', 'Carrier', 'Coverholder', 'Lob',
       'State', 'Product', 'RaterVersion', 'IsMasterPolicy',
       'MasterReferenceNumber', 'IsTestPolicy', 'UISchemaVersion',
       'AgentEffectiveDate', 'NoOfClaims', 'IsCoverageSaved',
       'AdjustPremiumType', 'AdjustPremiumReason', 'AdjustPremiumOtherReason',
       'CanAgentApproveEnd', 'QuoteValidity.Value', 'QuoteValidity.ValueType',
       'RenewalConfigurations.AfterForUnderwriter.Value',
       'RenewalConfigurations.AfterForUnderwriter.ValueType',
       'RenewalConfigurations.BeforeForUnderwriter.Value',
       'RenewalConfigurations.BeforeForUnderwriter.ValueType',
       'RenewalConfigurations.AfterForAgent.Value',
       'RenewalConfigurations.AfterForAgent.ValueType',
       'RenewalConfigurations.BeforeForAgent.Value',
       'RenewalConfigurations.BeforeForAgent.ValueType', 'Subjectivity'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 187, Finished, Available, Finished)

In [26]:
mergeandinsert('Attributes',data,['Policy_ref'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 188, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/Attributes_versions


In [27]:
data_lob

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 189, Finished, Available, Finished)

""


In [28]:
mergeandinsert('Attributes_Lob',data_lob,['Policy_ref','LobId'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 190, Finished, Available, Finished)

Skip since empty


In [29]:
mergeandinsert('Attributes_States',data_states,['Policy_ref','States_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 191, Finished, Available, Finished)

Skip since empty


### Payments

In [30]:
data_list = []
unique_id = 0
for i in landing_data:
    ref = policy_ref(i)
    if 'Payments' in i and i['Payments']:
        for j in i['Payments']:
            unique_id += 1
            j = {'Policy_ref': ref, 'Id': unique_id, **j}
            data_list.append(j)
    else:
        unique_id += 1
        j = {'Policy_ref': ref, 'Id': unique_id}
        data_list.append(j)

data = pd.json_normalize(data_list)


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 192, Finished, Available, Finished)

In [31]:
if data.empty==False:
    mergeandinsert('Payments',data,['Policy_ref','Id'])
else:
    pass

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 193, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Id` = df.`Id`


Merged new data into Tables/Payments_versions


### PolicyCoverages

In [32]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    Coverages = 0
    if i['PolicyCoverages']:
        for j in i['PolicyCoverages']['Coverages']:
            Coverages += 1
            j = {'Policy_ref': ref, 'Coverages_Id': Coverages, **j}
            data_list.append(j)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 194, Finished, Available, Finished)

In [33]:
mergeandinsert('PolicyCoverages',data,['Policy_ref','Coverages_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 195, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Coverages_Id` = df.`Coverages_Id`


Merged new data into Tables/PolicyCoverages_versions


### FeesAndTaxes

In [34]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    FeesAndTaxes = 0
    if i['FeesAndTaxes']:
        for j in i['FeesAndTaxes']:
            FeesAndTaxes += 1
            j = {'Policy_ref': ref, 'FeesAndTaxes_Id': FeesAndTaxes, **j}
            data_list.append(j)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 196, Finished, Available, Finished)

,Policy_ref,FeesAndTaxes_Id,Code,Description,Type,IsOverride,Status,Value,ValueType,Amount,AnnualAmount,ProductFeesAndTaxes,Id,Name,CreatedOn,CreatedBy,UpdatedOn,UpdatedBy,ProductFeesAndTaxes.Description
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,FASTFEE,2023 FIGA Assessment Fee,Fee,False,Active,0.01,P,20.64,20.64,nan,10.0,2023 FIGA Assessment Fee,2024-08-16T10:02:47.7886721+00:00,msingh,2025-08-07T10:01:10.8711179+00:00,asalunkhe,None
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2,MGAFEE,MGA Fee,Fee,False,Active,25.00,V,25.00,25.00,nan,7.0,MGA Fee,2024-07-01T11:37:13.0229228Z,msingh,2025-08-07T10:01:10.8710982+00:00,asalunkhe,None
2,111b5c8b-30b3-427f-977b-fe5ccd358b5f,3,EMPATRF,EMPA Trust Fund,Fee,False,Active,2.00,V,2.00,2.00,nan,11.0,EMPA Trust Fund,2024-08-16T10:02:47.7886727+00:00,msingh,2025-08-07T10:01:10.8711207+00:00,asalunkhe,None
3,111b5c8b-30b3-427f-977b-fe5ccd358b5f,4,SETUPFEE,Setup Fee,Fee,False,Active,0.00,V,0.00,0.00,nan,12.0,Setup Fee,2024-08-16T10:02:47.7886735+00:00,msingh,2025-08-07T10:01:10.8711236+00:00,asalunkhe,None
4,111b5c8b-30b3-427f-977b-fe5ccd358b5f,5,LFMAD,LFMA Discount,Fee,False,Active,-5.16,V,-5.16,-5.16,nan,13.0,LFMA Discount,2024-10-04T07:22:24.6795849+00:00,msingh,2025-08-07T10:01:10.8711265+00:00,asalunkhe,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,4,LPTD,LPT Discount,Fee,False,Active,0.00,V,0.00,0.00,nan,15.0,LPT Discount,2024-10-04T07:22:24.6795864+00:00,msingh,2025-08-07T10:01:10.8711335+00:00,asalunkhe,None
756,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,5,LPTD-MGA,LPT Discount - MGA,Fee,False,InActive,0.00,V,0.00,0.00,nan,16.0,LPT Discount - MGA,2024-10-04T07:22:24.679587+00:00,msingh,2025-08-07T10:01:10.8711369+00:00,asalunkhe,None
757,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,6,LFMAD,LFMA Discount,Fee,False,Active,0.00,V,0.00,0.00,nan,13.0,LFMA Discount,2024-10-04T07:22:24.6795849+00:00,msingh,2025-08-07T10:01:10.8711265+00:00,asalunkhe,None
758,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,7,LFMADMGA,LFMA Discount - MGA,Fee,False,InActive,0.00,V,0.00,0.00,nan,14.0,LFMA Discount - MGA,2024-10-04T07:22:24.6795855+00:00,msingh,2025-08-07T10:01:10.8711297+00:00,asalunkhe,None


In [35]:
# Please review the columns

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 197, Finished, Available, Finished)

In [36]:
try:
       data=data.loc[:,['Policy_ref', 'FeesAndTaxes_Id', 'Code', 'Description', 'Type',
       'IsOverride', 'Status', 'Value', 'ValueType', 'Amount', 'AnnualAmount',
       'IsProRated']]
except:
       data=data.reindex(columns=['Policy_ref', 'FeesAndTaxes_Id', 'Code', 'Description', 'Type',
       'IsOverride', 'Status', 'Value', 'ValueType', 'Amount', 'AnnualAmount',
       'IsProRated'])
data

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 198, Finished, Available, Finished)

,Policy_ref,FeesAndTaxes_Id,Code,Description,Type,IsOverride,Status,Value,ValueType,Amount,AnnualAmount,IsProRated
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,FASTFEE,2023 FIGA Assessment Fee,Fee,False,Active,0.01,P,20.64,20.64,NaN
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2,MGAFEE,MGA Fee,Fee,False,Active,25.00,V,25.00,25.00,NaN
2,111b5c8b-30b3-427f-977b-fe5ccd358b5f,3,EMPATRF,EMPA Trust Fund,Fee,False,Active,2.00,V,2.00,2.00,NaN
3,111b5c8b-30b3-427f-977b-fe5ccd358b5f,4,SETUPFEE,Setup Fee,Fee,False,Active,0.00,V,0.00,0.00,NaN
4,111b5c8b-30b3-427f-977b-fe5ccd358b5f,5,LFMAD,LFMA Discount,Fee,False,Active,-5.16,V,-5.16,-5.16,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
755,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,4,LPTD,LPT Discount,Fee,False,Active,0.00,V,0.00,0.00,NaN
756,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,5,LPTD-MGA,LPT Discount - MGA,Fee,False,InActive,0.00,V,0.00,0.00,NaN
757,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,6,LFMAD,LFMA Discount,Fee,False,Active,0.00,V,0.00,0.00,NaN
758,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,7,LFMADMGA,LFMA Discount - MGA,Fee,False,InActive,0.00,V,0.00,0.00,NaN


In [37]:
data['Amount']=data['Amount'].astype(float)

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 199, Finished, Available, Finished)

In [38]:
mergeandinsert('FeesAndTaxes',data,['Policy_ref','FeesAndTaxes_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 200, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`FeesAndTaxes_Id` = df.`FeesAndTaxes_Id`
Merged new data into Tables/FeesAndTaxes_versions


### TotalPremium

In [39]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    if i['TotalPremium']:
        j = i['TotalPremium']
        j = {'Policy_ref': ref, **j}
        data_list.append(j)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 201, Finished, Available, Finished)

,Policy_ref,BasicPremium,Surcharge,Discount,MinPremium,EffectivePremium,AnnualPremium,PriorAnnualPremium,CommutativeEffectivePremium,MonthlyEffectivePremium,FullyEarnedPremium,WorkingMonthlyPremium,Fees,Taxes,AnnualFees,AnnualTax,EffectivePremiumWithFeesAndTaxes,AnnualPremiumWithFeesAndTaxes
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2064,0,0,0,2064.0,2064,0,2064.0,0.00,0,0.00,5.86,0,5.86,0,2069.86,2069.86
1,ae46254b-2338-44ed-8238-486145c927fc,2999,0,0,0,2999.0,2999,0,2999.0,0.00,0,0.00,-3.49,0,-3.49,0,2995.51,2995.51
2,794deccd-0359-4ca3-b72f-a9d9ead11391,3745,0,0,0,3745.0,3745,0,3745.0,0.00,0,0.00,-10.95,0,-10.95,0,3734.05,3734.05
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,8291,0,0,0,-8291.0,8291,8291,0.0,-690.92,0,0.00,56.41,0,-56.41,0,-8234.59,8234.59
4,06748658-94cb-49fa-83e5-6a36813b41fd,1499,0,0,0,1499.0,1499,0,1499.0,0.00,0,0.00,11.51,0,11.51,0,1510.51,1510.51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,2999,0,0,0,2999.0,2999,0,2999.0,0.00,0,0.00,56.99,0,56.99,0,3055.99,3055.99
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,970,0,0,0,970.0,970,0,970.0,0.00,0,0.00,16.79,0,16.79,0,986.79,986.79
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,2570,0,0,0,2570.0,2570,0,2570.0,0.00,0,0.00,52.70,0,52.70,0,2622.70,2622.70
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,982,0,0,0,982.0,982,0,982.0,0.00,0,0.00,36.82,0,36.82,0,1018.82,1018.82


In [40]:
try:
    if data['OOSPremium']== np.nan:
        data['OOSPremium']=0
except:
    data['OOSPremium']=0
data['OOSPremium']=data['OOSPremium'].astype(float)    

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 202, Finished, Available, Finished)

In [41]:
mergeandinsert('TotalPremium',data,['Policy_ref'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 203, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`


Merged new data into Tables/TotalPremium_versions


### Premium

In [42]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    if i['Premium']:
        j = i['Premium']
        j = {'Policy_ref': ref, **j}
        data_list.append(j)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 204, Finished, Available, Finished)

,Policy_ref,EffectivePremium,AnnualPremium,Surcharge,Discount,AnnualSurcharge,AnnualDiscount,FullyEarnedPremium
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,0,0,0,0,0,0,0
1,ae46254b-2338-44ed-8238-486145c927fc,0,0,0,0,0,0,0
2,794deccd-0359-4ca3-b72f-a9d9ead11391,0,0,0,0,0,0,0
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,0,0,0,0,0,0,0
4,06748658-94cb-49fa-83e5-6a36813b41fd,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,0,0,0,0,0,0,0
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,0,0,0,0,0,0,0
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,0,0,0,0,0,0,0
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,0,0,0,0,0,0,0


In [43]:
# Please add Fully Earned Premium

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 205, Finished, Available, Finished)

In [44]:
try:
       data=data.loc[:,['Policy_ref', 'EffectivePremium', 'AnnualPremium', 'Surcharge','Discount', 'AnnualSurcharge', 'AnnualDiscount','FullyEarnedPremium']]
except:
       data=data.reindex(columns=['Policy_ref', 'EffectivePremium', 'AnnualPremium', 'Surcharge','Discount', 'AnnualSurcharge', 'AnnualDiscount','FullyEarnedPremium']).fillna(' ')

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 206, Finished, Available, Finished)

In [45]:
mergeandinsert('Premium',data,['Policy_ref'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 207, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/Premium_versions


### TransactionTotalPremium

In [46]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    if i['TransactionTotalPremium']:
        j = i['TransactionTotalPremium']
        j = {'Policy_ref': ref, **j}
        data_list.append(j)

df = pd.json_normalize(data_list)
df


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 208, Finished, Available, Finished)

,Policy_ref,BasicPremium,Surcharge,Discount,MinPremium,EffectivePremium,AnnualPremium,PriorAnnualPremium,CommutativeEffectivePremium,MonthlyEffectivePremium,FullyEarnedPremium,WorkingMonthlyPremium
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2064,0,0,0,2064.0,2064,0,2064.0,NaN,0,NaN
1,ae46254b-2338-44ed-8238-486145c927fc,2999,0,0,0,2999.0,2999,0,2999.0,NaN,0,NaN
2,794deccd-0359-4ca3-b72f-a9d9ead11391,3745,0,0,0,3745.0,3745,0,3745.0,NaN,0,NaN
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,8291,0,0,0,-8291.0,8291,8291,0.0,-690.92,0,0.00
4,06748658-94cb-49fa-83e5-6a36813b41fd,1499,0,0,0,1499.0,1499,0,1499.0,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,2999,0,0,0,2999.0,2999,0,2999.0,NaN,0,NaN
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,970,0,0,0,970.0,970,0,970.0,NaN,0,NaN
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,2570,0,0,0,2570.0,2570,0,2570.0,NaN,0,NaN
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,982,0,0,0,982.0,982,0,982.0,NaN,0,NaN


In [47]:
try:
    df=df.loc[:,['Policy_ref', 'BasicPremium', 'Surcharge', 'Discount', 'MinPremium',
       'EffectivePremium', 'AnnualPremium', 'PriorAnnualPremium',
       'CommutativeEffectivePremium', 'MonthlyEffectivePremium',
       'FullyEarnedPremium', 'WorkingMonthlyPremium']]
except:
    df=df.reindex(columns=['Policy_ref', 'BasicPremium', 'Surcharge', 'Discount', 'MinPremium',
       'EffectivePremium', 'AnnualPremium', 'PriorAnnualPremium',
       'CommutativeEffectivePremium', 'MonthlyEffectivePremium',
       'FullyEarnedPremium', 'WorkingMonthlyPremium'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 209, Finished, Available, Finished)

In [48]:
mergeandinsert('TransactionTotalPremium',df,['Policy_ref'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 210, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`


Merged new data into Tables/TransactionTotalPremium_versions


### Forms

In [49]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    Forms = 0
    if i['Forms']:
        for j in i['Forms']:
            Forms += 1
            j = {'Policy_ref': ref, 'Forms_Id': Forms, **j}
            data_list.append(j)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 211, Finished, Available, Finished)

,Policy_ref,Forms_Id,Status,FormName,FormDesc,Sequence,FormType,Template,IsMandatory,IsChecked,AcordCode,File,Dmspath,BlobLocation
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,,SFI FL PC 09 21,Policy Coversheet,1,DynamicDocx,iyiapo1z6omfc9b,True,True,EM055,,,nan
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2,,SFI FL HO WEL 06 23,Welcome Letter,2,DynamicDocx,6d4f3jtuf9omfkt,True,True,EM074,,,nan
2,111b5c8b-30b3-427f-977b-fe5ccd358b5f,3,,SFI FL HO5-B DEC 07 24,Policy Declarations,3,DynamicDocx,43ztkbm6w3omfcf,True,True,EM056,,,nan
3,111b5c8b-30b3-427f-977b-fe5ccd358b5f,4,,SFI FL HO5 RDD 10 23,Roof Deductible Disclosure Statement,4,DynamicDocx,yyj3mtnoudm,False,True,EM065,,,nan
4,111b5c8b-30b3-427f-977b-fe5ccd358b5f,5,,SFI FL HO5-B OTL 07 24,Comprehensive Homeowners Policy Outline of Cov...,5,DynamicDocx,ztsjl3tjxar,True,True,EM011,,,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1058,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,21,,OIR-B1-1655 02 10,Notice of Premium Discounts for Hurricane Loss...,38,DynamicDocx,tjmiexeieym,False,False,EM046,,,nan
1059,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,22,,SFI FL HO5 CRT 02 24,Change to Claims Reporting Timeline,39,Static,srhm0kx5alg,False,False,EM007,,,nan
1060,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,23,,SFI FL HO5 CTC 07 23,Cancellation Timeline Change,40,DynamicDocx,rsvlvnzsgut,False,False,EM005,,,nan
1061,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,24,,SFI FL HO5 AOB 01 23,Assignment of Benefits Prohibition Endorsement,41,Static,be4jelh2mcg,False,False,EM004,,,nan


In [50]:
mergeandinsert('Forms',data,['Policy_ref','Forms_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 212, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Forms_Id` = df.`Forms_Id`


Merged new data into Tables/Forms_versions


### PreviousPolicies

In [51]:
data_list = []
for i in landing_data:
    ref = policy_ref(i)
    if i['PreviousPolicies']:
        j = i['PreviousPolicies']
        j = {'Policy_ref': ref, **j}
        data_list.append(j)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 213, Finished, Available, Finished)

,Policy_ref,IsPreviousPolicy,PreviousPolicyNumber,OtherCoverages,IsPreviousClaims,NoOfClaims,TotalAmountClaimed
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,,,,,,
1,ae46254b-2338-44ed-8238-486145c927fc,,,,,,
2,794deccd-0359-4ca3-b72f-a9d9ead11391,,,,,,
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,,,,,,
4,06748658-94cb-49fa-83e5-6a36813b41fd,,,,,,
...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,,,,,,
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,,,,,,
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,,,,,,
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,,,,,,


In [52]:
mergeandinsert('PreviousPolicies',data,['Policy_ref'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 214, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/PreviousPolicies_versions


### Payplan

In [53]:
data_payplan_list = []
data_payplan_installment_list = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    payplan_id = 0
    if 'Payplan' in jsonn and jsonn['Payplan']:
        for json_data in jsonn['Payplan']:
            payplan_id += 1
            payplan_installment_id = 0

            if json_data.get('Installments'):
                for installments in json_data['Installments']:
                    payplan_installment_id += 1
                    installments = {'Policy_ref': ref, 'Payplan_Id': payplan_id, 'PayplanInstallments_Id': payplan_installment_id, **installments}
                    data_payplan_installment_list.append(installments)

            json_data = {'Policy_ref': ref, 'Payplan_Id': payplan_id, **json_data}    
            data_payplan_list.append(json_data)
    else:
        payplan_id += 1
        d = {'Policy_ref': ref, 'Payplan_Id': payplan_id}
        data_payplan_list.append(d)

data_payplan = pd.json_normalize(data_payplan_list)
data_payplan_installment = pd.json_normalize(data_payplan_installment_list)

for cols in data_payplan.columns:
    if data_payplan[cols].isna().sum() == data_payplan.shape[0]:
        data_payplan[cols] = data_payplan[cols].astype(str)

data_payplan.replace({np.nan: None}, inplace=True)
data_payplan


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 215, Finished, Available, Finished)

,Policy_ref,Payplan_Id,DownPaymentPercentage,DownPaymentCode,DownPaymentDueDays,DownPaymentAmount,PayInFull,NoOfInstallments,MoreThanDPAmount,IsSelected,PPDescription,Installments
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,40,Q40,2025-09-22,831.46,,3.0,,true,Quarterly 40% Dn,"[{'InstallmentNumber': 0, 'BillingPlan': 'Samp..."
1,ae46254b-2338-44ed-8238-486145c927fc,1,100,FP100,2025-09-24,2995.51,,None,,true,Full-Pay 100% Dn,[]
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,100,FP100,2025-09-24,3734.05,,None,,true,Full-Pay 100% Dn,[]
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,100,FP100,2025-07-09,8234.59,,None,,true,Full-Pay 100% Dn,[]
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,20,M8,2025-09-26,311.31,,10.0,,true,Monthly,"[{'InstallmentNumber': 0, 'BillingPlan': 'Samp..."
...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,20,M8,2025-10-15,656.79,,10.0,,true,Monthly,"[{'InstallmentNumber': 0, 'BillingPlan': 'Samp..."
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,20,M8,2025-09-15,210.79,,10.0,,true,Monthly,"[{'InstallmentNumber': 0, 'BillingPlan': 'Samp..."
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,100,FP100,2025-10-15,2622.7,,None,,true,Full-Pay 100% Dn,[]
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,100,FP100,2025-10-29,1018.82,,None,,true,Full-Pay 100% Dn,[]


In [54]:
try:
       data_payplan=data_payplan.loc[:,['Policy_ref', 'Payplan_Id', 'DownPaymentPercentage', 'DownPaymentCode',
       'DownPaymentDueDays', 'DownPaymentAmount', 'PayInFull',
       'NoOfInstallments', 'MoreThanDPAmount', 'IsSelected', 'PPDescription']]
except:
       data_payplan=data_payplan.reindex(columns=['Policy_ref', 'Payplan_Id', 'DownPaymentPercentage', 'DownPaymentCode',
       'DownPaymentDueDays', 'DownPaymentAmount', 'PayInFull',
       'NoOfInstallments', 'MoreThanDPAmount', 'IsSelected', 'PPDescription']).fillna(' ')

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 216, Finished, Available, Finished)

In [55]:
data_payplan.isna().sum()

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 217, Finished, Available, Finished)

Policy_ref                0
Payplan_Id                0
DownPaymentPercentage     0
DownPaymentCode           0
DownPaymentDueDays        0
DownPaymentAmount         0
PayInFull                 0
NoOfInstallments         45
MoreThanDPAmount          0
IsSelected                0
PPDescription             0
dtype: int64

In [56]:
mergeandinsert('Payplan',data_payplan,['Policy_ref','Payplan_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 218, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Payplan_Id` = df.`Payplan_Id`
Merged new data into Tables/Payplan_versions


In [57]:
mergeandinsert('Payplan_Installments',data_payplan_installment,['Policy_ref','Payplan_Id','PayplanInstallments_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 219, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Payplan_Id` = df.`Payplan_Id` AND old_data.`PayplanInstallments_Id` = df.`PayplanInstallments_Id`
Merged new data into Tables/Payplan_Installments_versions


### ExternalRefrence

In [58]:
data_list = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    ExternalRefrences_Id = 0
    if jsonn.get('ExternalRefrences'):
        for json_data in jsonn['ExternalRefrences']:
            ExternalRefrences_Id += 1
            json_data = {'Policy_ref': ref, 'ExternalRefrences_Id': ExternalRefrences_Id, **json_data}
            data_list.append(json_data)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 220, Finished, Available, Finished)

,Policy_ref,ExternalRefrences_Id,Id,ReferenceNumber,ReferenceTarget,Status
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,7797,2564,CORE_LOGIC_POLICY_ID,Active
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2,None,100001032,BillingAccountNumber,Active
2,ae46254b-2338-44ed-8238-486145c927fc,1,7820,1672,CORE_LOGIC_POLICY_ID,Active
3,ae46254b-2338-44ed-8238-486145c927fc,2,None,100001069,BillingAccountNumber,Active
4,794deccd-0359-4ca3-b72f-a9d9ead11391,1,7815,1668,CORE_LOGIC_POLICY_ID,Active
...,...,...,...,...,...,...
185,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,2,None,100001106,BillingAccountNumber,Active
186,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,8546,1872,CORE_LOGIC_POLICY_ID,Active
187,31e77022-e1fa-4725-a7b1-d8c2fd500b85,2,None,100001107,BillingAccountNumber,Active
188,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,1,8546,1872,CORE_LOGIC_POLICY_ID,Active


In [59]:
mergeandinsert('ExternalRefrences',data,['Policy_ref','ExternalRefrences_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 221, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`ExternalRefrences_Id` = df.`ExternalRefrences_Id`


Merged new data into Tables/ExternalRefrences_versions


### PriorInsurances

In [60]:
# Remove the library import

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 222, Finished, Available, Finished)

In [61]:
import pandas as pd
import numpy as np

# Initialize an empty list to store PriorInsurances data
data_list = []

# Iterate through the landing_data JSON
for jsonn in landing_data:
    ref = policy_ref(jsonn)  # Extract the policy reference for each entry
    PriorInsurances_Id = 0

    # Check if 'PriorInsurances' exists and is not empty
    if 'PriorInsurances' in jsonn and isinstance(jsonn['PriorInsurances'], dict):
        PriorInsurances_Id += 1

        # Create a new dictionary for each entry in PriorInsurances
        json_data = {'Policy_ref': ref, 'PriorInsurances_Id': PriorInsurances_Id, **jsonn['PriorInsurances']}

        # Append the data to the list
        data_list.append(json_data)
    else:
        # Handle missing PriorInsurances case by inserting just the Policy_ref and PriorInsurances_Id
        PriorInsurances_Id += 1
        json_data = {'Policy_ref': ref, 'PriorInsurances_Id': PriorInsurances_Id}
        data_list.append(json_data)

# Normalize the list of data and create a DataFrame
data = pd.json_normalize(data_list)

# Convert columns with all missing values to string type (if needed)
for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

# Replace NaN with None (if needed for further processing)
data.replace({np.nan: None}, inplace=True)

# Output the DataFrame
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 223, Finished, Available, Finished)

,Policy_ref,PriorInsurances_Id,IsCurrentlyInsured,CompanyName,OtherCompanyName,EffectiveDate,ExpirationDate,Premium,Status,LapsesInCoverages,ReasonLapsesInCoverages
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,False,,,,,0,,0,
1,ae46254b-2338-44ed-8238-486145c927fc,1,False,,,,,0,,0,
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,False,,,,,0,,0,
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,False,,,,,0,,0,
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,False,,,,,0,,0,
...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,False,,,,,0,,0,
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,False,,,,,0,,0,
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,False,,,,,0,,0,
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,False,,,,,0,,0,


In [62]:
mergeandinsert('PriorInsurances',data,['Policy_ref','PriorInsurances_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 224, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`PriorInsurances_Id` = df.`PriorInsurances_Id`
Merged new data into Tables/PriorInsurances_versions


### Claims

In [63]:
data_list = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)  # Extract the policy reference for each entry
    Claims_Id = 0
    
    if jsonn.get('Claims'):
        for json_data in jsonn['Claims']:
            Claims_Id += 1
            json_data = {'Policy_ref': ref, 'Claims_Id': Claims_Id, **json_data}
            data_list.append(json_data)

# Normalize the list of data and create a DataFrame
data = pd.json_normalize(data_list)

# Convert columns with all missing values to string type (if needed)
for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

# Replace NaN with None (if needed for further processing)
data.replace({np.nan: None}, inplace=True)

# Output the DataFrame
data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 225, Finished, Available, Finished)

""


In [64]:
mergeandinsert('Claims',data,['Policy_ref','Claims_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 226, Finished, Available, Finished)

Skip since empty


### PremiumFactors

In [65]:
data_list = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    PremiumFactors_Id = 0
    if 'PremiumFactors' in jsonn and jsonn['PremiumFactors']:
        for json_data in jsonn['PremiumFactors']:
            PremiumFactors_Id += 1
            json_data = {'Policy_ref': ref, 'PremiumFactors_Id': PremiumFactors_Id, **json_data}
            data_list.append(json_data)

data = pd.json_normalize(data_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)

data


StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 227, Finished, Available, Finished)

,Policy_ref,PremiumFactors_Id,Name,Type,Description,Rate,Limit,LimitAmount,Premium,AnnualPremium,Status,IsApplicable,IsSelected,IsMandatory,Deductible,EffectivePremium,PremiumDifference
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0
1,ae46254b-2338-44ed-8238-486145c927fc,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0
4,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,2,ProRate,,Pro Rate Factor,1.000,100.0,100.0,0,0,Active,True,True,False,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0
155,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0
156,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0
157,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,1,RiskPrem,,Risk Premium,0.000,0.0,0.0,0,0,Active,True,True,False,0,0,0


In [66]:
mergeandinsert('PremiumFactors',data,['Policy_ref','PremiumFactors_Id'])

StatementMeta(, ac961dfc-9aed-44da-ac11-9e09a7063eb1, 228, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`PremiumFactors_Id` = df.`PremiumFactors_Id`


Merged new data into Tables/PremiumFactors_versions


### Transaction

In [67]:
import copy
import pandas as pd
import numpy as np

data_list = []
data_trans_list = []

for i in landing_data:
    ref = policy_ref(i)
    trans_id = 0
    
    if 'Transaction' in i and i['Transaction']:
        j = copy.deepcopy(i['Transaction'])  # Deep copy added
        
        # Pop internal node before flattening
        trans_hist = j.pop('TransactionStatusHistory', None)
        if trans_hist:
            trans_id += 1
            datas = {'Policy_ref': ref, 'TransactionStatusHistory_Id': trans_id, **trans_hist}
            data_trans_list.append(datas)
        
        j = {'Policy_ref': ref, **j}
        data_list.append(j)

data = pd.json_normalize(data_list)
data_trans = pd.json_normalize(data_trans_list)

for cols in data.columns:
    if data[cols].isna().sum() == data.shape[0]:
        data[cols] = data[cols].astype(str)

data.replace({np.nan: None}, inplace=True)

data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 18, Finished, Available, Finished)

,Policy_ref,Rewrite,Date,EffectiveDate,Type,Status,Number,IsOutOfSequence,RequestedBy,RequestedById,...,RenewalOffers.RenewalOfferFlags.UpdatedBy,RenewalOffers.RenewalOfferFlags.EmailSentFlag,Verification.IsInsuredESignVerified,Verification.IsAgentESignVerified,Verification.IsEmailVerified,Verification.IsOTPVerified,Verification.InsuredOTP,EffectiveTime,SCR,ReasonId
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,,2025-09-22,2025-09-22,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan
1,ae46254b-2338-44ed-8238-486145c927fc,,2025-09-24,2025-09-24,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan
2,794deccd-0359-4ca3-b72f-a9d9ead11391,,2025-09-24,2025-09-24,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,,2025-09-24,2025-07-09,Cancellation,Committed,1,False,,,...,,False,False,False,False,False,,12:01 AM,0.1,nan
4,06748658-94cb-49fa-83e5-6a36813b41fd,,2025-09-26,2025-09-26,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,,2025-10-15,2025-10-15,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,,2025-10-15,2025-09-15,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,,2025-10-15,2025-10-15,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,,2025-10-15,2025-11-27,Policy,Committed,0,False,,,...,,False,False,False,False,False,,None,None,nan


In [68]:
mergeandinsert('Transaction',data,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 22, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`


Merged new data into Tables/Transaction_versions


In [69]:
mergeandinsert('Transaction_TransactionStatusHistory',data_trans,['Policy_ref','TransactionStatusHistory_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 21, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`TransactionStatusHistory_Id` = df.`TransactionStatusHistory_Id`


Merged new data into Tables/Transaction_TransactionStatusHistory_versions


### Agency

In [70]:
# data_list = []
# data_comm_list = []
# for i in landing_data:
#     ref = policy_ref(i)
#     trans_id = 0
#     if 'Agency' in i and i['Agency']:
#         j = i['Agency']
        
#         if 'Communications' in j and j['Communications']:
#             for datas in j['Communications']:
#                 trans_id += 1
#                 datas = {'Policy_ref': ref, 'Communications_Id': trans_id, **datas}
#                 data_comm_list.append(datas)
#         else:
#             trans_id += 1
#             datas = {'Policy_ref': ref, 'Communications_Id': trans_id}
#             data_comm_list.append(datas)
        
#         j = {'Policy_ref': ref, **j}
#         data_list.append(j)

# data = pd.json_normalize(data_list)
# data_comm = pd.json_normalize(data_comm_list)

# data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 23, Finished, Available, Finished)

In [71]:
# Please review Reference and Product , plus few details missing like Role , Fee

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 24, Finished, Available, Finished)

In [72]:
# try:
#     data=data.loc[:,['Policy_ref', 'Client', 'Code', 'Name', 'Status', 'Reference',
#         'Products', 'Address.Number', 'Address.AddressType',
#        'Address.Description', 'Address.AddressLine1', 'Address.AddressLine2',
#        'Address.AptSuite', 'Address.City', 'Address.County',
#        'Address.CountyCode', 'Address.State', 'Address.PoBox',
#        'Address.Postalcode', 'Address.FormattedAddress',
#        'Address.UnFormattedAddress', 'Address.AdministrationArea1',
#        'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
#        'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
#        'Address.Territory', 'Address.Business', 'Address.PlaceId',
#        'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName']]
# except:
#     data=data.reindex(columns=['Policy_ref', 'Client', 'Code', 'Name', 'Status','Reference', 
#        'Products', 'Address.Number', 'Address.AddressType',
#        'Address.Description', 'Address.AddressLine1', 'Address.AddressLine2',
#        'Address.AptSuite', 'Address.City', 'Address.County',
#        'Address.CountyCode', 'Address.State', 'Address.PoBox',
#        'Address.Postalcode', 'Address.FormattedAddress',
#        'Address.UnFormattedAddress', 'Address.AdministrationArea1',
#        'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
#        'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
#        'Address.Territory', 'Address.Business', 'Address.PlaceId',
#        'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName']).fillna(' ')
# for i in data.columns:
#     if i =='Reference' or i =='Products':
#         data[i]=np.array(data[i])
#     elif data[i].isna().sum()==data.shape[0]:
#         data[i]=data[i].astype(str)
# data       

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 25, Finished, Available, Finished)

In [73]:
# mergeandinsert('Agency',data,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 26, Finished, Available, Finished)

In [74]:
# data.dtypes

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 27, Finished, Available, Finished)

In [75]:
# data_comm

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 28, Finished, Available, Finished)

In [76]:
# mergeandinsert('Agency_Communications',data_comm,['Policy_ref','Communications_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 29, Finished, Available, Finished)

In [77]:
data_agency_list = []
data_comm_list = []
data_addr_list = []
data_ref_list = []
data_products_list = []

for i in landing_data:
    ref = policy_ref(i)

    if 'Agency' in i and i['Agency']:
        j = i['Agency']

        # --- Communications ---
        comms = j.get('Communications', []) or []
        # if comms is None or not isinstance(comms, list):
        #     print(f"⚠️ Bad 'Communications' format for Policy_ref {ref}: {type(comms)} → {comms}")
        #     # Skip to next agency record
        #     continue
        comm_id = 0
        for datas in comms:
            comm_id += 1
            if isinstance(datas, dict):
                datas = {'Policy_ref': ref, 'Communications_Id': comm_id, **datas}
            else:
                datas = {'Policy_ref': ref, 'Communications_Id': comm_id, 'Value': datas}
            data_comm_list.append(datas)

        # --- Address (single object instead of list) ---
        addr = j.get('Address') or []
        if isinstance(addr, dict):
            addr_row = {'Policy_ref': ref, 'Address_Id': 1, **addr}
            data_addr_list.append(addr_row)
        elif addr:  # non-null but not dict (edge case)
            addr_row = {'Policy_ref': ref, 'Address_Id': 1, 'AddressValue': addr}
            data_addr_list.append(addr_row)

        # --- Reference (list of strings here) ---
        refs = j.get('Reference', []) or []
        ref_id = 0
        for datas in refs:
            ref_id += 1
            if isinstance(datas, dict):
                datas = {'Policy_ref': ref, 'Reference_Id': ref_id, **datas}
            else:  # string or other type
                datas = {'Policy_ref': ref, 'Reference_Id': ref_id, 'Reference': datas}
            data_ref_list.append(datas)

        # --- Products (list of strings) ---
        products = j.get('Products', []) or []
        prod_id = 0
        for datas in products:
            prod_id += 1
            if isinstance(datas, dict):
                datas = {'Policy_ref': ref, 'Product_Id': prod_id, **datas}
            else:
                datas = {'Policy_ref': ref, 'Product_Id': prod_id, 'Products': datas}
            data_products_list.append(datas)

        # --- Remaining Agency (clean parent without arrays) ---
        clean_agency = {k: v for k, v in j.items() if k not in ['Communications', 'Reference', 'Products']}
        data_agency_list.append({'Policy_ref': ref, **clean_agency})

# --- Create DataFrames ---
df_agency = pd.json_normalize(data_agency_list)
df_comm = pd.json_normalize(data_comm_list)
df_addr = pd.json_normalize(data_addr_list)
df_ref = pd.json_normalize(data_ref_list)
df_products = pd.json_normalize(data_products_list)

# Replace NaN with None
for df in [df_agency, df_comm, df_addr, df_ref, df_products]:
    df.replace({np.nan: None}, inplace=True)


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 30, Finished, Available, Finished)

In [78]:
# Parent Agency (no arrays)
mergeandinsert('Agency', df_agency, ['Policy_ref'])

# Communications child
mergeandinsert('Agency_Communications', df_comm, ['Policy_ref', 'Communications_Id'])

# Address child
mergeandinsert('Agency_Address', df_addr, ['Policy_ref', 'Address_Id'])

# Reference (AIM block) child
mergeandinsert('Agency_Reference', df_ref, ['Policy_ref', 'Reference_Id'])


# Products child
mergeandinsert('Agency_Products', df_products, ['Policy_ref', 'Product_Id'])


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 31, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`


Merged new data into Tables/Agency_versions
Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Communications_Id` = df.`Communications_Id`


Merged new data into Tables/Agency_Communications_versions


Created new Delta table at Tables/Agency_Address_versions
Created new Delta table at Tables/Agency_Reference_versions
Created new Delta table at Tables/Agency_Products_versions


### Agent

In [79]:
# Similar as Agency

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 32, Finished, Available, Finished)

In [80]:
# data_list = []
# data_comm_list = []
# data_rd_list = []
# data_rd_r_list = []

# for i in landing_data:
#     ref = policy_ref(i)
#     trans_id = 0
#     ref_id = 0

#     if 'Agent' in i and i['Agent']:
#         j = i['Agent']

#         # Process Communications
#         if 'Communications' in j and j['Communications']:
#             for datas in j['Communications']:
#                 trans_id += 1
#                 datas = {'Policy_ref': ref, 'Communications_Id': trans_id, **datas}
#                 data_comm_list.append(datas)
#         else:
#             trans_id += 1
#             datas = {'Policy_ref': ref, 'Communications_Id': trans_id}
#             data_comm_list.append(datas)

#         # Process RestrictionsDetails
#         try:
#             if 'RestrictionsDetails' in j and j['RestrictionsDetails']:
#                 r_id = 0
#                 for restriction in j['RestrictionsDetails']:
#                     ref_id += 1
#                     restriction = {'Policy_ref': ref, 'Id': ref_id, **restriction}
#                     data_rd_list.append(restriction)

#                     # Process Restrictions inside each RestrictionsDetail
#                     if 'Restrictions' in restriction and restriction['Restrictions']:
#                         for res in restriction['Restrictions']:
#                             r_id += 1
#                             res_data = {'Policy_ref': ref, 'Restriction_Id': r_id, **res}
#                             data_rd_r_list.append(res_data)
#         except Exception as e:
#             print(f"Error processing RestrictionsDetails for Policy_ref {ref}: {e}")
#             restriction = {'Policy_ref': ref, 'Id': ref_id}
#             data_rd_list.append(restriction)

#         # Process other Agent fields
#         j = {'Policy_ref': ref, **j}
#         data_list.append(j)

# data = pd.json_normalize(data_list)
# data_comm = pd.json_normalize(data_comm_list)
# data_rd = pd.json_normalize(data_rd_list)
# data_rd_r = pd.json_normalize(data_rd_r_list)

# data.replace({pd.NA: None}, inplace=True)

# data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 33, Finished, Available, Finished)

In [81]:
# data.columns

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 34, Finished, Available, Finished)

In [82]:
# try:
#        data=data.loc[:,['Policy_ref', 'Client', 'Code', 'Name', 'Status','Reference',
#               'Products', 
#               'IsLicenseAgreementAccepted', 
#               'StartDate',
#               'LocationName',  
#               'Address.Number',
#               'Address.AddressType', 'Address.Description',
#               'Address.AddressLine1', 'Address.AddressLine2', 'Address.AptSuite',
#               'Address.City', 'Address.County', 'Address.CountyCode', 'Address.State',
#               'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
#               'Address.UnFormattedAddress', 'Address.AdministrationArea1',
#               'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
#               'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
#               'Address.Territory', 'Address.Business', 'Address.PlaceId',
#               'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName']]
# except:
#        data=data.reindex(columns=['Policy_ref', 'Client', 'Code', 'Name', 'Status', 'Reference',
#               'Products', 
#               'IsLicenseAgreementAccepted', 
#               'StartDate',
#               'LocationName',  
#               'Address.Number',
#               'Address.AddressType', 'Address.Description',
#               'Address.AddressLine1', 'Address.AddressLine2', 'Address.AptSuite',
#               'Address.City', 'Address.County', 'Address.CountyCode', 'Address.State',
#               'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
#               'Address.UnFormattedAddress', 'Address.AdministrationArea1',
#               'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
#               'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
#               'Address.Territory', 'Address.Business', 'Address.PlaceId',
#               'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName']).fillna(' ')

              
# data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 35, Finished, Available, Finished)

In [83]:
# mergeandinsert('Agent',data,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 36, Finished, Available, Finished)

In [84]:
# data_comm

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 37, Finished, Available, Finished)

In [85]:
# mergeandinsert('Agent_Communications',data_comm,['Policy_ref','Communications_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 38, Finished, Available, Finished)

In [86]:
# data_rd.columns

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 39, Finished, Available, Finished)

In [87]:
# try:
#     data_rd=data_rd.loc[:,['Policy_ref', 'Id', 'EffectiveDate', 'ExpiryDate', 'Message', 'Name',
#        'Product']]
# except:
#     data_rd=data_rd.reindex(columns=['Policy_ref', 'Id', 'EffectiveDate', 'ExpiryDate', 'Message', 'Name',
#        'Product'])


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 40, Finished, Available, Finished)

In [88]:
# data_rd_r

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 41, Finished, Available, Finished)

In [89]:
# mergeandinsert('Agent_RestrictionDetails_Restrictions',data_rd_r,['Policy_ref','Restriction_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 42, Finished, Available, Finished)

In [90]:
data_agent_list = []
data_comm_list = []
data_addr_list = []
data_ref_list = []
data_products_list = []

for i in landing_data:
    ref = policy_ref(i)

    if 'Agent' in i and i['Agent']:
        j = i['Agent']

        # --- Communications ---
        comms = j.get('Communications', []) or []
        comm_id = 0
        for datas in comms:
            comm_id += 1
            if isinstance(datas, dict):
                datas = {'Policy_ref': ref, 'Communications_Id': comm_id, **datas}
            else:
                datas = {'Policy_ref': ref, 'Communications_Id': comm_id, 'Value': datas}
            data_comm_list.append(datas)

        # --- Address (single object instead of list) ---
        addr = j.get('Address')
        if isinstance(addr, dict):
            addr_row = {'Policy_ref': ref, 'Address_Id': 1, **addr}
            data_addr_list.append(addr_row)
        elif addr:  # non-null but not dict (edge case)
            addr_row = {'Policy_ref': ref, 'Address_Id': 1, 'AddressValue': addr}
            data_addr_list.append(addr_row)

        # --- Reference (list of strings here) ---
        refs = j.get('Reference', [])
        ref_id = 0
        for datas in refs:
            ref_id += 1
            if isinstance(datas, dict):
                datas = {'Policy_ref': ref, 'Reference_Id': ref_id, **datas}
            else:  # string or other type
                datas = {'Policy_ref': ref, 'Reference_Id': ref_id, 'Reference': datas}
            data_ref_list.append(datas)

        # --- Products (list of strings) ---
        products = j.get('Products', [])
        prod_id = 0
        for datas in products:
            prod_id += 1
            if isinstance(datas, dict):
                datas = {'Policy_ref': ref, 'Product_Id': prod_id, **datas}
            else:
                datas = {'Policy_ref': ref, 'Product_Id': prod_id, 'Products': datas}
            data_products_list.append(datas)

        # --- Remaining Agent (clean parent without arrays) ---
        clean_agent = {k: v for k, v in j.items() if k not in ['Communications', 'Reference', 'Products']}
        data_agent_list.append({'Policy_ref': ref, **clean_agent})

# --- Create DataFrames ---
df_agent = pd.json_normalize(data_agent_list)
df_comm = pd.json_normalize(data_comm_list)
df_addr = pd.json_normalize(data_addr_list)
df_ref = pd.json_normalize(data_ref_list)
df_products = pd.json_normalize(data_products_list)

# Replace NaN with None
for df in [df_agent, df_comm, df_addr, df_ref, df_products]:
    df.replace({np.nan: None}, inplace=True)


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 43, Finished, Available, Finished)

In [91]:
# Parent Agent (no arrays)
mergeandinsert('Agent', df_agent, ['Policy_ref'])

# Communications child
mergeandinsert('Agent_Communications', df_comm, ['Policy_ref', 'Communications_Id'])

# Address child
mergeandinsert('Agent_Address', df_addr, ['Policy_ref', 'Address_Id'])

# Reference (AIM block) child
mergeandinsert('Agent_Reference', df_ref, ['Policy_ref', 'Reference_Id'])


# Products child
mergeandinsert('Agent_Products', df_products, ['Policy_ref', 'Product_Id'])


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 44, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/Agent_versions
Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Communications_Id` = df.`Communications_Id`


Merged new data into Tables/Agent_Communications_versions


Created new Delta table at Tables/Agent_Address_versions
Created new Delta table at Tables/Agent_Reference_versions


Created new Delta table at Tables/Agent_Products_versions


### AgentCommission

In [92]:
# Please review ~YY

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 45, Finished, Available, Finished)

In [93]:
# Review Done ~MR

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 46, Finished, Available, Finished)

In [94]:
data_list = []

for i in landing_data:
    ref = policy_ref(i)
    id = 1
    
    if 'AgentCommission' in i and i['AgentCommission']:
        ac_data = i.get('AgentCommission') or []
        
        for entry in ac_data:
            if not isinstance(entry, dict):
                print(f"⚠️ Non-dict AgentCommission entry for Policy_ref={ref}: {entry} (type={type(entry)})")
                continue  # skip this malformed entry
            print(entry)
            j = {'Policy_ref': ref, 'AgentCommission_Id': id, **entry}
            id += 1
            data_list.append(j)
    else:
        j = {'Policy_ref': ref, 'AgentCommission_Id': id}
        data_list.append(j)

data = pd.json_normalize(data_list)
data.replace({np.nan: None}, inplace=True)


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 47, Finished, Available, Finished)

{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'NEW'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'REN'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'NEW'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'REN'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'NEW'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'REN'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'NEW'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'REN'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'NEW'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'REN'}
{'Value': '12', 'ValueType': 'Percent', 'CommissionType': '', 'TransactionType': 'NEW'}
{'Value': '12', 'ValueType': 'Pe

In [95]:
mergeandinsert('AgentCommission',data,['Policy_ref','AgentCommission_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 48, Finished, Available, Finished)

Created new Delta table at Tables/AgentCommission_versions


### Underwriting

In [96]:
data_list = []

for i in landing_data:
    ref = policy_ref(i)
    trans_id = 0
    for datas in i['Underwriting']:
        try:    
            trans_id += 1
            datas = {'Policy_ref': ref, 'Id': trans_id, **datas}
            data_list.append(datas)
        except:
            pass       

data = pd.json_normalize(data_list)

data.replace({np.nan: None}, inplace=True)

data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 49, Finished, Available, Finished)

""


In [97]:
mergeandinsert('Underwriting',data,['Policy_ref','Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 50, Finished, Available, Finished)

Skip since empty


### UnderWriter

In [98]:
# Similar to Agent and Agency

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 51, Finished, Available, Finished)

In [99]:
data = []
data_comm = []
data_products = []

for i in landing_data:
    ref = policy_ref(i)
    trans_id = 0
    products_id = 0
    if 'UnderWriter' in i and i['UnderWriter']:
        j = i['UnderWriter']

        communications = j.get('Communications', [])
        if communications:
            trans_id = 0
            for datas in communications:
                trans_id += 1
                datas = {'Policy_ref': ref, 'Communications_Id': trans_id, **datas}
                data_comm.append(datas)
        else:
            datas = {'Policy_ref': ref,'Communications_Id': trans_id}
            data_comm.append(datas)

        prod = j.get('Products', [])
        if prod:
            products_id = 0
            for product in prod:
                products_id += 1
                datas = {'Policy_ref': ref, 'Products_Id': products_id, 'Product_Code': product}
                data_products.append(datas)
        else:
            datas = {'Policy_ref': ref,'Products_Id': products_id}
            data_products.append(datas)

        j = {'Policy_ref': ref, **j}
        data.append(j)

data = pd.json_normalize(data)
data_comm = pd.json_normalize(data_comm)
data_products = pd.json_normalize(data_products)

for i in data.columns:
    if data[i].isna().sum() == data.shape[0]:
        data[i] = data[i].astype(str)

data.replace({np.nan: None}, inplace=True)

data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 52, Finished, Available, Finished)

,Policy_ref,Client,Code,Name,Status,Reference,Products,Role,Fee,JWTstring,...,Details.Contact.PolicyEmailId,Details.Contact.FromEmailId,Details.Contact.EmailCCId,Details.Contact.PreferedContactType,Details.Contact.SecondaryEmailId,UWAttributes.UWOffice,UWAttributes.ContractNumber,UWAttributes.AccountExecCode,Details,UWAttributes
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,,,Embark UW Dept,Active,[],[],Underwriter,None,,...,newhomeuw@embarkgeneral.com,noreply@cogitate.us,newhomeuw@embarkgeneral.com,E,newhomeuw@embarkgeneral.com,,,,nan,nan
1,ae46254b-2338-44ed-8238-486145c927fc,EMBARK,3975,Mayank UW,Active,[],[SFEMHO5FL],Underwriter,None,None,...,None,None,None,None,None,None,None,None,nan,nan
2,794deccd-0359-4ca3-b72f-a9d9ead11391,EMBARK,3975,Mayank UW,Active,[],[SFEMHO5FL],Underwriter,None,None,...,None,None,None,None,None,None,None,None,nan,nan
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,EMBARK,3975,Mayank UW,Active,[],[SFEMHO5FL],Underwriter,None,None,...,None,None,None,None,None,None,None,None,nan,nan
4,06748658-94cb-49fa-83e5-6a36813b41fd,,,Embark UW Dept,Active,[],[],Underwriter,None,,...,newhomeuw@embarkgeneral.com,noreply@cogitate.us,newhomeuw@embarkgeneral.com,E,newhomeuw@embarkgeneral.com,,,,nan,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,EMBARK,3975,Mayank UW,Active,[],[SFEMHO5FL],Underwriter,None,None,...,None,None,None,None,None,None,None,None,nan,nan
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,EMBARK,3975,Mayank UW,Active,[],[SFEMHO5FL],Underwriter,None,None,...,None,None,None,None,None,None,None,None,nan,nan
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,EMBARK,3975,Mayank UW,Active,[],[SFEMHO5FL],Underwriter,None,None,...,None,None,None,None,None,None,None,None,nan,nan
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,EMBARK,3975,Mayank UW,Active,[],[SFEMHO5FL],Underwriter,None,None,...,None,None,None,None,None,None,None,None,nan,nan


In [100]:
try:
       data=data.loc[:,['Policy_ref', 'Client', 'Code', 'Name', 'Status', 
       #'Reference', 
       #'Products',
        'Address.Number', 'Address.AddressType',
       'Address.IsManual', 'Address.Description', 'Address.AddressLine1',
       'Address.AddressLine2', 'Address.AptSuite', 'Address.City',
       'Address.County', 'Address.CountyCode', 'Address.State',
       'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
       'Address.UnFormattedAddress', 'Address.AdministrationArea1',
       'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
       'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
       'Address.Territory', 'Address.Business', 'Address.PlaceId',
       #'Address.AptSuiteLot', 
       'Address.Street', 'Address.StreetName',
       'UWAttributes.UWOffice', 'UWAttributes.ContractNumber',
       'UWAttributes.AccountExecCode']]
except:
       data=data.reindex(columns=['Policy_ref', 'Client', 'Code', 'Name', 'Status', 
       #'Reference', 
       #'Products', 
       'Address.Number', 'Address.AddressType',
       'Address.IsManual', 'Address.Description', 'Address.AddressLine1',
       'Address.AddressLine2', 'Address.AptSuite', 'Address.City',
       'Address.County', 'Address.CountyCode', 'Address.State',
       'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
       'Address.UnFormattedAddress', 'Address.AdministrationArea1',
       'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
       'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
       'Address.Territory', 'Address.Business', 'Address.PlaceId',
       #'Address.AptSuiteLot',
        'Address.Street', 'Address.StreetName',
       'UWAttributes.UWOffice', 'UWAttributes.ContractNumber',
       'UWAttributes.AccountExecCode']).fillna(' ')

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 53, Finished, Available, Finished)

In [101]:
mergeandinsert('UnderWriter',data,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 54, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/UnderWriter_versions


In [102]:
mergeandinsert('UnderWriter_Communications',data_comm,['Policy_ref','Communications_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 55, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Communications_Id` = df.`Communications_Id`
Merged new data into Tables/UnderWriter_Communications_versions


In [103]:
mergeandinsert('UnderWriter_Products',data_products,['Policy_ref','Products_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 56, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Products_Id` = df.`Products_Id`


Merged new data into Tables/UnderWriter_Products_versions


### Policy

In [104]:
# New
data = []
for i in landing_data:
    ref = policy_ref(i)
    j = {'Policy_ref': ref,**i}
    data.append(j)

data = pd.json_normalize(data)
data

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 57, Finished, Available, Finished)

,Policy_ref
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f
1,ae46254b-2338-44ed-8238-486145c927fc
2,794deccd-0359-4ca3-b72f-a9d9ead11391
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7
4,06748658-94cb-49fa-83e5-6a36813b41fd
...,...
90,c831e867-b099-4812-909a-febb8c464d40
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85


In [105]:
# Old
# data = []
# for i in landing_data:
#     ref = policy_ref(i)
#     try:
#         j = {'Policy_ref': ref, "CompanyId": "40", **i}
#         data.append(j)
#     except:
#         j = {'Policy_ref': ref}
#         data.append(j)

# data = pd.json_normalize(data)
# data

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 58, Finished, Available, Finished)

In [106]:
# Please review the column name to make sure we did not miss any attributes

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 59, Finished, Available, Finished)

In [107]:
try:
    data=data.loc[:,["Policy_ref","id","AccountId","QuoteNumber",
"QuoteVersion",
"isInactiveQuoteVersion",
"IsPolicyBind",
"IsInForce","IsRenewed","PolicyNumber",
"LinkedPolicyNumber",
"PolicyTerm.Value","PolicyTerm.ValueType","EffectiveDate",
"ExpirationDate","PolicyStatus" ,"CurrentVersion" ,"CurrentVersionEffectiveFrom" ,"PayMode" ,"InceptionTime" ,"SignatureType" 
,"BindDate","QuoteDate" ,"QuoteExpDate" ,"RenewalTerm","IsOOS","RenewalQuoteID","Audit.CreatedBy" ,
"Audit.CreatedOn","Audit.DeletedOn" ,"Audit.LastUpdatedBy" ,"Audit.LastUpdatedOn","PreviousPolicies.IsPreviousClaims","PolicyStatusRemarks",
"PreviousPolicies.IsPreviousPolicy","PreviousPolicies.NoOfClaims","PreviousPolicies.OtherCoverages" ,
"PreviousPolicies.PreviousPolicyNumber" ,"PreviousPolicies.TotalAmountClaimed",
"ReconciliationFlags.LiabilityPolicy" ,"ReconciliationFlags.ClueCount","ReconciliationFlags.ClueVioationStatus" ,
"ReconciliationFlags.ClueReferenceNumber" ,
"ReconciliationFlags.ClueOrderStatusCode",
"_rid" ,"_self" ,
"_etag" ,"_attachments" ,"_ts"]]
except:
    data=data.reindex(columns=["Policy_ref","id","AccountId","QuoteNumber",
"QuoteVersion",
"isInactiveQuoteVersion",
"IsPolicyBind",
"IsInForce","IsRenewed","PolicyNumber",
"LinkedPolicyNumber",
"PolicyTerm.Value","PolicyTerm.ValueType","EffectiveDate",
"ExpirationDate","PolicyStatus" ,"CurrentVersion" ,"CurrentVersionEffectiveFrom" ,"PayMode" ,"InceptionTime" ,"SignatureType" 
,"BindDate","QuoteDate" ,"QuoteExpDate" ,"RenewalTerm","IsOOS","RenewalQuoteID","Audit.CreatedBy" ,
"Audit.CreatedOn","Audit.DeletedOn" ,"Audit.LastUpdatedBy" ,"Audit.LastUpdatedOn","PreviousPolicies.IsPreviousClaims","PolicyStatusRemarks",
"PreviousPolicies.IsPreviousPolicy","PreviousPolicies.NoOfClaims","PreviousPolicies.OtherCoverages" ,
"PreviousPolicies.PreviousPolicyNumber" ,"PreviousPolicies.TotalAmountClaimed",
"ReconciliationFlags.LiabilityPolicy" ,"ReconciliationFlags.ClueCount","ReconciliationFlags.ClueVioationStatus" ,
"ReconciliationFlags.ClueReferenceNumber" ,
"ReconciliationFlags.ClueOrderStatusCode",
"_rid" ,"_self" ,
"_etag" ,"_attachments" ,"_ts"])
for i in data.columns:
    if data[i].isna().sum()==data.shape[0]:
        data[i]=data[i].astype(str)
data.replace({np.nan:None},inplace=True)
data



StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 60, Finished, Available, Finished)

,Policy_ref,id,AccountId,QuoteNumber,QuoteVersion,isInactiveQuoteVersion,IsPolicyBind,IsInForce,IsRenewed,PolicyNumber,...,ReconciliationFlags.LiabilityPolicy,ReconciliationFlags.ClueCount,ReconciliationFlags.ClueVioationStatus,ReconciliationFlags.ClueReferenceNumber,ReconciliationFlags.ClueOrderStatusCode,_rid,_self,_etag,_attachments,_ts
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,ae46254b-2338-44ed-8238-486145c927fc,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,794deccd-0359-4ca3-b72f-a9d9ead11391,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
4,06748658-94cb-49fa-83e5-6a36813b41fd,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [108]:
data

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 61, Finished, Available, Finished)

,Policy_ref,id,AccountId,QuoteNumber,QuoteVersion,isInactiveQuoteVersion,IsPolicyBind,IsInForce,IsRenewed,PolicyNumber,...,ReconciliationFlags.LiabilityPolicy,ReconciliationFlags.ClueCount,ReconciliationFlags.ClueVioationStatus,ReconciliationFlags.ClueReferenceNumber,ReconciliationFlags.ClueOrderStatusCode,_rid,_self,_etag,_attachments,_ts
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,ae46254b-2338-44ed-8238-486145c927fc,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,794deccd-0359-4ca3-b72f-a9d9ead11391,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
4,06748658-94cb-49fa-83e5-6a36813b41fd,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [109]:
mergeandinsert('Policy',data,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 62, Finished, Available, Finished)

Created new Delta table at Tables/Policy_versions


### JointPolicyHolders

In [110]:
data = []
data_comm = []
for i in landing_data:
    ref = policy_ref(i)
    JointPolicyHolders_Id = 0
    if i['JointPolicyHolders']:
        for j in i['JointPolicyHolders']:
            trans_id = 0
            JointPolicyHolders_Id += 1
            try:
                for datas in j['Communications']:
                    trans_id += 1
                    datas = {'Policy_ref': ref, 'JointPolicyHolders_Id': JointPolicyHolders_Id, 'Communications_Id': trans_id, **datas}
                    data_comm.append(datas)
            except:
                datas = {'Policy_ref': ref, "JointPolicyHolders_Id": JointPolicyHolders_Id, 'Communications_Id': 1}
                data_comm.append(datas)
            j = {'Policy_ref': ref, 'JointPolicyHolders_Id': JointPolicyHolders_Id, **j}
            data.append(j)

data = pd.json_normalize(data)
data_comm = pd.json_normalize(data_comm)

data.replace({np.nan: None}, inplace=True)

data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 63, Finished, Available, Finished)

,Policy_ref,JointPolicyHolders_Id,Id,FirstName,MiddleName,LastName,DateOfBirth,InsuredType,Occupation,IsHighProfile,...,Address.FormattedAddress,Address.UnFormattedAddress,Address.Status,Address.AptSuite,Address.PoBox,Address.CityCode,Address.Territory,Address.TerritoryCode,Age.Value,Age.ValueType
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,,,,,,,,False,...,,,,,,,,,0,
1,ae46254b-2338-44ed-8238-486145c927fc,1,,,,,,,,False,...,,,,,,,,,0,
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,,,,,,,,False,...,,,,,,,,,0,
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,,,,,,,,False,...,,,,,,,,,0,
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,,,,,,,,False,...,,,,,,,,,0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,,,,,,,,False,...,,,,,,,,,0,
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,,,,,,,,False,...,,,,,,,,,0,
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,,,,,,,,False,...,,,,,,,,,0,
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,,,,,,,,False,...,,,,,,,,,0,


In [111]:
# Please review

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 64, Finished, Available, Finished)

In [112]:
try:
       data=data.loc[:,['Policy_ref', 'JointPolicyHolders_Id', 'FirstName', 'MiddleName',
       'LastName', 'DateOfBirth', 'InsuredType', 'Occupation', 'IsHighProfile',
       'IsAddressSameAsRisk', 'Age.Value', 'Age.ValueType',
       'Address.Number', 'Address.AddressType', 'Address.Description',
       'Address.AddressLine1', 'Address.AddressLine2', 'Address.AptSuite',
       'Address.City', 'Address.County', 'Address.CountyCode', 'Address.State',
       'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
       'Address.UnFormattedAddress', 'Address.AdministrationArea1',
       'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
       'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
       'Address.Territory', 'Address.Business', 'Address.PlaceId',
       'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName']]
except:
       data=data.reindex(columns=['Policy_ref', 'JointPolicyHolders_Id', 'FirstName', 'MiddleName',
       'LastName', 'DateOfBirth', 'InsuredType', 'Occupation', 'IsHighProfile',
       'IsAddressSameAsRisk', 'Age.Value', 'Age.ValueType',
       'Address.Number', 'Address.AddressType', 'Address.Description',
       'Address.AddressLine1', 'Address.AddressLine2', 'Address.AptSuite',
       'Address.City', 'Address.County', 'Address.CountyCode', 'Address.State',
       'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
       'Address.UnFormattedAddress', 'Address.AdministrationArea1',
       'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
       'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
       'Address.Territory', 'Address.Business', 'Address.PlaceId',
       'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName'])
data       

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 65, Finished, Available, Finished)

,Policy_ref,JointPolicyHolders_Id,FirstName,MiddleName,LastName,DateOfBirth,InsuredType,Occupation,IsHighProfile,IsAddressSameAsRisk,...,Address.Long,Address.Name,Address.Premise,Address.Status,Address.Territory,Address.Business,Address.PlaceId,Address.AptSuiteLot,Address.Street,Address.StreetName
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
1,ae46254b-2338-44ed-8238-486145c927fc,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,,,,,,,False,False,...,,,NaN,,,NaN,,NaN,NaN,


In [113]:
data.columns

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 66, Finished, Available, Finished)

Index(['Policy_ref', 'JointPolicyHolders_Id', 'FirstName', 'MiddleName',
       'LastName', 'DateOfBirth', 'InsuredType', 'Occupation', 'IsHighProfile',
       'IsAddressSameAsRisk', 'Age.Value', 'Age.ValueType', 'Address.Number',
       'Address.AddressType', 'Address.Description', 'Address.AddressLine1',
       'Address.AddressLine2', 'Address.AptSuite', 'Address.City',
       'Address.County', 'Address.CountyCode', 'Address.State',
       'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
       'Address.UnFormattedAddress', 'Address.AdministrationArea1',
       'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
       'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
       'Address.Territory', 'Address.Business', 'Address.PlaceId',
       'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName'],
      dtype='object')

In [114]:
mergeandinsert('JointPolicyHolders',data,['Policy_ref','JointPolicyHolders_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 67, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`JointPolicyHolders_Id` = df.`JointPolicyHolders_Id`


Merged new data into Tables/JointPolicyHolders_versions


In [115]:
data_comm

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 68, Finished, Available, Finished)

,Policy_ref,JointPolicyHolders_Id,Communications_Id,Type,SubType,Value,Status
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,1,PhNo,Primary,,
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,2,Email,Primary,,
2,ae46254b-2338-44ed-8238-486145c927fc,1,1,PhNo,Primary,,
3,ae46254b-2338-44ed-8238-486145c927fc,1,2,Email,Primary,,
4,794deccd-0359-4ca3-b72f-a9d9ead11391,1,1,PhNo,Primary,,
...,...,...,...,...,...,...,...
185,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,2,Email,Primary,,
186,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,1,PhNo,Primary,,
187,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,2,Email,Primary,,
188,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,1,1,PhNo,Primary,,


This cell is commented because: AnalysisException: [DELTA_EMPTY_DATA] Data used in creating the Delta table doesn't have any columns.

### InsuredAccount`

In [116]:
data = []
data_comm = []
data_buss = []
data_loc = []

for i in landing_data:
    ref = policy_ref(i)
    InsuredAccount_Id = 0
    if i['InsuredAccount']:
        j = i['InsuredAccount']
        comm_id = 0
        loc_id = 0
        InsuredAccount_Id += 1

        if j['BusinessInfo']:
            buss = j['BusinessInfo']
            try:
                for loc in j['BusinessInfo']['Locations']:
                    loc_id += 1
                    loc = {'Policy_ref': ref, 'InsuredAccount_Id': InsuredAccount_Id, 'Location_Id': loc_id, **loc}
                    data_loc.append(loc)
            except:
                loc = {'Policy_ref': ref, "InsuredAccount_Id": InsuredAccount_Id, 'Location_Id': 1}
                data_loc.append(loc)

            buss = {'Policy_ref': ref, 'InsuredAccount_Id': InsuredAccount_Id, **buss}
            data_buss.append(buss)

        try:
            for datas in j['Communications']:
                comm_id += 1
                datas = {'Policy_ref': ref, 'InsuredAccount_Id': InsuredAccount_Id, 'Communications_Id': comm_id, **datas}
                data_comm.append(datas)
        except:
            datas = {'Policy_ref': ref, "InsuredAccount_Id": InsuredAccount_Id, 'Communications_Id': 1}
            data_comm.append(datas)

        j = {'Policy_ref': ref, 'InsuredAccount_Id': InsuredAccount_Id, **j}
        data.append(j)

data = pd.json_normalize(data)
data_comm = pd.json_normalize(data_comm)
data_buss = pd.json_normalize(data_buss)
data_loc = pd.json_normalize(data_loc)

data.replace({np.nan: None}, inplace=True)

data


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 69, Finished, Available, Finished)

,Policy_ref,InsuredAccount_Id,Type,CreationDate,UserName,FirstName,MiddleName,LastName,DisplayName,HasJointPolicyHolder,...,BusinessInfo.IndustryType,BusinessInfo.BusinessDecsription,BusinessInfo.AnnualRevenue,BusinessInfo.NumberOFLocations,BusinessInfo.Locations,BankDetails.AccountName,BankDetails.BankName,BankDetails.BranchName,BankDetails.AccountType,BankDetails.RoutingNo
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,,,,Nidhi,,Westwood 2,Nidhi Westwood 2,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
1,ae46254b-2338-44ed-8238-486145c927fc,1,,,,Nidhi,,Mortgagee,Nidhi Mortgagee,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,,,,Nidhi,,Payment,Nidhi Payment,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,,,,Tara,,Woc,Tara Woc,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,,,,Nidhi,,West 1,Nidhi West 1,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,,,,Nidhi,,Renewal 2,Nidhi Renewal 2,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,,,,Nidhi,,Renewal 1,Nidhi Renewal 1,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,,,,Nidhi,,Sr Renewal 3,Nidhi Sr Renewal 3,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,,,,Morty,,Singh,Morty Singh,false,...,,,0,0,"[{'NickName': '', 'LocationNumber': 0, 'IsVali...",,,,,


In [117]:
# Few Attributes Missing , plus review the column name seems to be wrong

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 70, Finished, Available, Finished)

In [118]:
try:
    data=data.loc[:,['Policy_ref', 'InsuredAccount_Id', 'Type', 'CreationDate', 'Suffix','UserName', 'FirstName', 'MiddleName', 'LastName', 'DisplayName', 'DOB', 
#'Communications', 
'IsAuthorizeForReports', 'IsAuthorizationforConsumerRatingInformation', 'IsAuthorizeForCreditScore', 
'IsVerifiedByLexisNexis', 
'BankDetails.AccountName', 'BankDetails.BankName', 'BankDetails.BranchName', 'BankDetails.AccountType', 'BankDetails.RoutingNo', 'Address.Number', 
'Address.AddressType', 'Address.IsManual', 'Address.Description', 'Address.AddressLine1', 'Address.AddressLine2', 'Address.AptSuite', 'Address.City', 
'Address.County', 'Address.CountyCode', 'Address.State', 'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress', 'Address.UnFormattedAddress', 
'Address.AdministrationArea1', 'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat', 'Address.Long', 'Address.Name', 'Address.Premise',
 'Address.Status', 'Address.Territory', 'Address.Business', 'Address.PlaceId', 'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName', 
 'CreditScore.Score', 'CreditScore.ScoreDescription', 'CreditScore.StatusCode', 'CreditScore.Message', 'CreditScore.NeedApiCall']]
except:
    data=data.reindex(columns=['Policy_ref', 'InsuredAccount_Id', 'Type', 'CreationDate', 'Suffix','UserName', 'FirstName', 'MiddleName', 'LastName', 'DisplayName', 'DOB', 
#'Communications', 
'IsAuthorizeForReports', 'IsAuthorizationforConsumerRatingInformation', 'IsAuthorizeForCreditScore', 
'IsVerifiedByLexisNexis', 
'BankDetails.AccountName', 'BankDetails.BankName', 'BankDetails.BranchName', 'BankDetails.AccountType', 'BankDetails.RoutingNo', 'Address.Number', 
'Address.AddressType', 'Address.IsManual', 'Address.Description', 'Address.AddressLine1', 'Address.AddressLine2', 'Address.AptSuite', 'Address.City', 
'Address.County', 'Address.CountyCode', 'Address.State', 'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress', 'Address.UnFormattedAddress', 
'Address.AdministrationArea1', 'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat', 'Address.Long', 'Address.Name', 'Address.Premise',
 'Address.Status', 'Address.Territory', 'Address.Business', 'Address.PlaceId', 'Address.AptSuiteLot', 'Address.Street', 'Address.StreetName', 
 'CreditScore.Score', 'CreditScore.ScoreDescription', 'CreditScore.StatusCode', 'CreditScore.Message', 'CreditScore.NeedApiCall'])
data 

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 71, Finished, Available, Finished)

,Policy_ref,InsuredAccount_Id,Type,CreationDate,Suffix,UserName,FirstName,MiddleName,LastName,DisplayName,...,Address.Business,Address.PlaceId,Address.AptSuiteLot,Address.Street,Address.StreetName,CreditScore.Score,CreditScore.ScoreDescription,CreditScore.StatusCode,CreditScore.Message,CreditScore.NeedApiCall
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,,,NaN,,Nidhi,,Westwood 2,Nidhi Westwood 2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ae46254b-2338-44ed-8238-486145c927fc,1,,,NaN,,Nidhi,,Mortgagee,Nidhi Mortgagee,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,,,NaN,,Nidhi,,Payment,Nidhi Payment,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,,,NaN,,Tara,,Woc,Tara Woc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,,,NaN,,Nidhi,,West 1,Nidhi West 1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,,,NaN,,Nidhi,,Renewal 2,Nidhi Renewal 2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,,,NaN,,Nidhi,,Renewal 1,Nidhi Renewal 1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,,,NaN,,Nidhi,,Sr Renewal 3,Nidhi Sr Renewal 3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,,,NaN,,Morty,,Singh,Morty Singh,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [119]:
try:
       data_buss=data_buss.loc[:,['Policy_ref', 'InsuredAccount_Id', 'BusinessType', 'YearsInBusiness',
       'IncorporationAge', 'BusinessName', 'DisplayName', 'RegistrationNumber',
       'BusinessStruct', 'SpecialRisk', 'NoOfOwners', 'FullTimeEmployees',
       'PartTimeEmployees', 'Website', 'IndustryType', 'BusinessDecsription',
       'AnnualRevenue', 'NumberOFLocations']]
except:
       data_buss

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 72, Finished, Available, Finished)

In [120]:
mergeandinsert('InsuredAccount',data,['Policy_ref','InsuredAccount_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 73, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`InsuredAccount_Id` = df.`InsuredAccount_Id`
Merged new data into Tables/InsuredAccount_versions


In [121]:
for i in data_comm.columns:
    if data_comm[i].isna().sum()==data_comm.shape[0]:
        data_comm[i]=data_comm[i].astype(str)
data_comm.replace({np.nan:None},inplace=True)
data_comm.head()

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 74, Finished, Available, Finished)

,Policy_ref,InsuredAccount_Id,Communications_Id,Type,SubType,Value,Status
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,1,PhNo,Primary,7231638412,
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,2,Email,Primary,nmehwala@cogitate.com,
2,ae46254b-2338-44ed-8238-486145c927fc,1,1,PhNo,Primary,9879876786,
3,ae46254b-2338-44ed-8238-486145c927fc,1,2,Email,Primary,nmehwala@cogitate.com,
4,794deccd-0359-4ca3-b72f-a9d9ead11391,1,1,PhNo,Primary,9286483268,


In [122]:
mergeandinsert('InsuredAccount_Communications',data_comm,['Policy_ref','InsuredAccount_Id','Communications_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 75, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`InsuredAccount_Id` = df.`InsuredAccount_Id` AND old_data.`Communications_Id` = df.`Communications_Id`
Merged new data into Tables/InsuredAccount_Communications_versions


In [123]:
mergeandinsert('InsuredAccount_BussinessInfo',data_buss,['Policy_ref','InsuredAccount_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 76, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`InsuredAccount_Id` = df.`InsuredAccount_Id`
Merged new data into Tables/InsuredAccount_BussinessInfo_versions


In [124]:
try:
       data_loc=data_loc.loc[:,['Policy_ref', 'InsuredAccount_Id', 'Location_Id', 'NickName',
       'LocationNumber', 'IsValid', 'Status', 'Type',
       'Address.Number', 'Address.AddressType', 'Address.Description',
       'Address.AddressLine1', 'Address.AddressLine2', 'Address.AptSuite',
       'Address.City', 'Address.County', 'Address.CountyCode', 'Address.State',
       'Address.PoBox', 'Address.Postalcode', 'Address.FormattedAddress',
       'Address.UnFormattedAddress', 'Address.AdministrationArea1',
       'Address.AdministrationArea2', 'Address.Locality', 'Address.Lat',
       'Address.Long', 'Address.Name', 'Address.Premise', 'Address.Status',
       'Address.Territory', 'Address.Business', 'Address.PlaceId',
       'Address.Street', 'Address.StreetName']]
except:
       data_loc

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 77, Finished, Available, Finished)

In [125]:
mergeandinsert('InsuredAccount_BussinessInfo_Locations',data_loc,['Policy_ref','InsuredAccount_Id','Location_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 78, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`InsuredAccount_Id` = df.`InsuredAccount_Id` AND old_data.`Location_Id` = df.`Location_Id`
Merged new data into Tables/InsuredAccount_BussinessInfo_Locations_versions


### Risks

##### Property

In [126]:
df_property = []
df_propertyriskattributes = []
df_propertypremium = []
df_propertyadditionalparties = []
df_propertypremiumfactors = []
df_propertycoverages = []
df_propertyaddress = []
df_propertyscheduledpersonalprop = []
df_propertgolf = []
df_propertyriskattributes_dogdetails=[]
df_propertyriskattributes_golf=[]
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    if jsonn['Risks']['Properties']:
        for data in jsonn['Risks']['Properties']:
            if "UnitId" not in data:
                print(f"Skipping policy {ref}, property has no UnitId")
                continue  
            property_id = data["UnitId"]

            try:
                if data['Premium']:
                    propertypremium = {'Policy_ref': ref, 'Property_Id': property_id, **data['Premium']}
                    df_propertypremium.append(propertypremium)
            except:
                propertypremium = {'Policy_ref': ref, 'Property_Id': property_id}
                df_propertypremium.append(propertypremium)

            try:
                if data['PropertyRiskAttributes']:
                    if data['PropertyRiskAttributes']['DogDetails']:
                        did=1
                        for i in data['PropertyRiskAttributes']['DogDetails']:
                            d={'Policy_ref':ref,'DD_id':did,**i}
                            df_propertyriskattributes_dogdetails.append(d)
                            did+=1
                    if data['PropertyRiskAttributes']['GolfCartDetails']:
                        gid=1
                        for i in data['PropertyRiskAttributes']['GolfCartDetails']:
                            d={'Policy_ref':ref,'GCD_Id':gid,**i}
                            df_propertyriskattributes_golf.append(d)
                            gid+=1
                    propertyriskattributes = {'Policy_ref': ref, 'Property_Id': property_id, **data['PropertyRiskAttributes']}
                    df_propertyriskattributes.append(propertyriskattributes)
            except:
                propertyriskattributes = {'Policy_ref': ref, 'Property_Id': property_id}
                df_propertyriskattributes.append(propertyriskattributes)

            try:
                if data['Address']:
                    propertyaddress = {'Policy_ref': ref, 'Property_Id': property_id, **data['Address']}
                    df_propertyaddress.append(propertyaddress)
            except:
                propertyaddress = {'Policy_ref': ref, 'Property_Id': property_id}
                df_propertyaddress.append(propertyaddress)

            try:
                for pf_data in data['PremiumFactors']:
                    pf_data = {'Policy_ref': ref, 'Property_Id': property_id, **pf_data}
                    df_propertypremiumfactors.append(pf_data)
            except:
                pf_data = {'Policy_ref': ref, 'Property_Id': property_id}
                df_propertypremiumfactors.append(pf_data)

            try:
                for cvgs_data in data['Coverages']:
                    cvgs_data = {'Policy_ref': ref, 'Property_Id': property_id, **cvgs_data}
                    df_propertycoverages.append(cvgs_data)
            except:
                cvgs_data = {'Policy_ref': ref, 'Property_Id': property_id}
                df_propertycoverages.append(cvgs_data)

            try:
                for spp_data in data['ScheduledPersonalProperty']:
                    spp_data = {'Policy_ref': ref, 'Property_Id': property_id, **spp_data}
                    df_propertyscheduledpersonalprop.append(spp_data)
            except:
                spp_data = {'Policy_ref': ref, 'Property_Id': property_id}
                df_propertyscheduledpersonalprop.append(spp_data)

            try:
                for gccd_data in data['GolfCartCoverageDetails']:
                    gccd_data = {'Policy_ref': ref, 'Property_Id': property_id, **gccd_data}
                    df_propertgolf.append(gccd_data)
            except:
                gccd_data = {'Policy_ref': ref, 'Property_Id': property_id}
                df_propertgolf.append(gccd_data)

            data = {'Policy_ref': ref, **data}
            df_property.append(data)

df_property = pd.json_normalize(df_property)
df_propertyriskattributes = pd.json_normalize(df_propertyriskattributes)
df_propertypremium = pd.json_normalize(df_propertypremium)
df_propertyadditionalparties = pd.json_normalize(df_propertyadditionalparties)
df_propertypremiumfactors = pd.json_normalize(df_propertypremiumfactors)
df_propertycoverages = pd.json_normalize(df_propertycoverages)
df_propertyaddress = pd.json_normalize(df_propertyaddress)
df_propertyscheduledpersonalprop = pd.json_normalize(df_propertyscheduledpersonalprop)
df_propertgolf = pd.json_normalize(df_propertgolf)
df_propertyriskattributes_dogdetails=pd.json_normalize(df_propertyriskattributes_dogdetails)
df_propertyriskattributes_golf=pd.json_normalize(df_propertyriskattributes_golf)
df_property.replace({np.nan: None}, inplace=True)


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 79, Finished, Available, Finished)

In [127]:
df_property

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 80, Finished, Available, Finished)

,Policy_ref,Community,ModelPlan,OtherModelPlan,PropertyAge,UnitId,UnitNumber,Name,UnitType,Status,...,Address.AddressType,Address.FormattedAddress,Address.UnFormattedAddress,Address.Status,Address.AptSuite,Address.PoBox,Address.CityCode,Address.Territory,Address.TerritoryCode,PropertyRiskAttributes.TIV
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,Coastal Cove,Model S,,0,1,1,,P,,...,p,,"2021 raulerson ct, apopka, FL 32721",,,,,,,None
1,ae46254b-2338-44ed-8238-486145c927fc,Celebration - R40,Hayden,,0,1,1,,P,,...,p,,"7410 barrier cove way, celebration, FL 34747",,,,,,,None
2,794deccd-0359-4ca3-b72f-a9d9ead11391,Emery,Aspen,,0,1,1,,P,,...,p,,"10729 sw estella ln, port st lucie, FL 34987",,,,,,,None
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,Coral Lago,Wheatley,,0,1,1,,P,,...,p,,"8480 nw 39th ct, coral springs, FL 33065",,,,,,,None
4,06748658-94cb-49fa-83e5-6a36813b41fd,Fox Pointe at Rivers Edge,Marigold,,0,1,1,,P,,...,p,,"28005 Poppy Court, Leesburg, FL 34748",,,,,,,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,Celebration - R40,Hayden,,0,1,1,,P,,...,p,,"7410 barrier cove way, celebration, FL 34747",,,,,,,None
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,RiverTown Gardens North RL 50s,Beecher,,0,1,1,,P,,...,p,,"892 orange branch trail, st johns, FL 32259",,,,,,,None
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,Celebration - T24,Anna Maria,,0,1,1,,P,,...,p,,"1833 coastal court, celebration, FL 34747",,,,,,,None
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,Addison Landing - D50,Cascades,,0,1,1,,P,,...,p,,"902 honey petal lane, deland, FL 32720",,,,,,,None


In [128]:
# Need Improvemnet , Builder Name and Code

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 81, Finished, Available, Finished)

In [129]:
try:
    df_property1=df_property.loc[:,["Policy_ref","PropertyAge","UnitId","UnitNumber","Name",
"UnitType","Status","Audit.CreatedBy","Audit.CreatedOn","Audit.LastUpdatedBy",
"Audit.LastUpdatedOn"]]
except:
    df_property1=df_property.reindex(columns=["Policy_ref","PropertyAge","UnitId","UnitNumber","Name",
"UnitType","Status","Audit.CreatedBy","Audit.CreatedOn","Audit.LastUpdatedBy",
"Audit.LastUpdatedOn"])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 82, Finished, Available, Finished)

In [130]:
print(df_propertyriskattributes.columns.tolist())

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 83, Finished, Available, Finished)

['Policy_ref', 'Property_Id', 'DTCGrouping', 'DwellingAge', 'AgeOfRoof', 'RCE', 'DescribeRoofType', 'RoofYear', 'SidingType', 'SalesPrice', 'EstimatedCloseDate', 'ReplacementCost', 'RoofSlope', 'SoffitType', 'RooftoWallAttachment', 'Homestyle', 'IsManualAddress', 'IsLLC', 'HasSameMailingAddress', 'OccupancyType', 'PropertyType', 'CoverageStartDate', 'YearBuilt', 'SquareFeet', 'ConstructionType', 'HydrantDistance', 'FireStationDistance', 'DistanceToTidalWater', 'FireStationDriveDuration', 'NoOfStories', 'NoOfFamilies', 'NoOfBedrooms', 'NoOfBaths', 'SecuredCommunity', 'Foundation', 'FireScore', 'FireAlarm', 'BurglarAlarms', 'SprinklerSystem', 'ShortTermRental', 'WeeksRentedPerYear', 'Deadbolt', 'FireExtinguisher', 'VisibleToNeighbours', 'RoofType', 'RoofShape', 'RoofUpdateType', 'RoofUpdateYear', 'WaterHeaterLocation', 'HeatingType', 'HeatingUpdateType', 'HeatingUpdateYear', 'ElectricalType', 'ElectricalUpdateType', 'ElectricalUpdateYear', 'PlumbingType', 'PlumbingUpdateType', 'PlumbingU

In [131]:
try:
    df_propertyriskattributes=df_propertyriskattributes.loc[:,['Policy_ref', 'Property_Id', 'DTCGrouping', 'DwellingAge', 'AgeOfRoof', 'RCE', 'DescribeRoofType', 'RoofYear', 'SidingType', 'SalesPrice', 'EstimatedCloseDate', 'ReplacementCost', 'RoofSlope', 'SoffitType', 'RooftoWallAttachment', 'Homestyle', 'IsManualAddress', 'IsLLC', 'HasSameMailingAddress', 'OccupancyType', 'PropertyType', 'CoverageStartDate', 'YearBuilt', 'SquareFeet', 'ConstructionType', 'HydrantDistance', 'FireStationDistance', 'DistanceToTidalWater', 'FireStationDriveDuration', 'NoOfStories', 'NoOfFamilies', 'NoOfBedrooms', 'NoOfBaths', 'SecuredCommunity', 'Foundation', 'FireScore', 'FireAlarm', 'BurglarAlarms', 'SprinklerSystem', 'ShortTermRental', 'WeeksRentedPerYear', 'Deadbolt', 'FireExtinguisher', 'VisibleToNeighbours', 'RoofType', 'RoofShape', 'RoofUpdateType', 'RoofUpdateYear', 'WaterHeaterLocation', 'HeatingType', 'HeatingUpdateType', 'HeatingUpdateYear', 'ElectricalType', 'ElectricalUpdateType', 'ElectricalUpdateYear', 'PlumbingType', 'PlumbingUpdateType', 'PlumbingUpdateYear', 'TIV', 'MinimumPremium', 'BrushFireDistance', 'DsValFireScore', 'DsValBrushFireDistance', 'CovBPercentageOfCovA', 'CovCPercentageOfCovA', 'CovDPercentageOfCovA', 'StormShutters', 'ACVRoofLossSettlement', 'ScreenEnclosureOrCarport', 'DistanceToCoast', 'BCEGID', 'CensusBlockGroup', 'CensusBlockVersion', 'RespondingFireDistrictName', 'RespondingFireDistrictId', 'WindRegion', 'WindTerrain', 'WindSpeedRegion0', 'RoofDeck', 'RoofCover', 'RoofCoverAgeClassification', 'PredomRoofMaterial', 'OpeningProtection', 'SecWaterResistance', 'SinkholeTerritory', 'HurricaneTerritory', 'AOPTerritory', 'WaterTerritory', 'CoverageAPremium', 'CoverageBPremium', 'CoverageCPremium', 'CoverageDPremium', 'CoverageEPremium', 'CoverageFPremium', 'SeniorDiscount', 'WindMitigationFeatures', 'PaperlessDiscount', 'WaterLeakDetectionCredit', 'RoofDeductibleCredit', 'RoofDeductibleCovA', 
    #'GolfCartDetails', 
    'noOfGolfCart', 
    #'DogDetails', 
    'NumberOfDogs', 'ProtectionClass.County', 'ProtectionClass.Protected', 'ProtectionClass.UnProtected', 'ProtectionClass.EffectiveDate', 'BCEGDetails.CommunityName', 'BCEGDetails.County', 'BCEGDetails.BCEGNumber', 'BCEGDetails.BeginYear']]
except:
    df_propertyriskattributes=df_propertyriskattributes.reindex(columns=['Policy_ref', 'Property_Id', 'DTCGrouping', 'DwellingAge', 'AgeOfRoof', 'RCE', 'DescribeRoofType', 'RoofYear', 'SidingType', 'SalesPrice', 'EstimatedCloseDate', 'ReplacementCost', 'RoofSlope', 'SoffitType', 'RooftoWallAttachment', 'Homestyle', 'IsManualAddress', 'IsLLC', 'HasSameMailingAddress', 'OccupancyType', 'PropertyType', 'CoverageStartDate', 'YearBuilt', 'SquareFeet', 'ConstructionType', 'HydrantDistance', 'FireStationDistance', 'DistanceToTidalWater', 'FireStationDriveDuration', 'NoOfStories', 'NoOfFamilies', 'NoOfBedrooms', 'NoOfBaths', 'SecuredCommunity', 'Foundation', 'FireScore', 'FireAlarm', 'BurglarAlarms', 'SprinklerSystem', 'ShortTermRental', 'WeeksRentedPerYear', 'Deadbolt', 'FireExtinguisher', 'VisibleToNeighbours', 'RoofType', 'RoofShape', 'RoofUpdateType', 'RoofUpdateYear', 'WaterHeaterLocation', 'HeatingType', 'HeatingUpdateType', 'HeatingUpdateYear', 'ElectricalType', 'ElectricalUpdateType', 'ElectricalUpdateYear', 'PlumbingType', 'PlumbingUpdateType', 'PlumbingUpdateYear', 'TIV', 'MinimumPremium', 'BrushFireDistance', 'DsValFireScore', 'DsValBrushFireDistance', 'CovBPercentageOfCovA', 'CovCPercentageOfCovA', 'CovDPercentageOfCovA', 'StormShutters', 'ACVRoofLossSettlement', 'ScreenEnclosureOrCarport', 'DistanceToCoast', 'BCEGID', 'CensusBlockGroup', 'CensusBlockVersion', 'RespondingFireDistrictName', 'RespondingFireDistrictId', 'WindRegion', 'WindTerrain', 'WindSpeedRegion0', 'RoofDeck', 'RoofCover', 'RoofCoverAgeClassification', 'PredomRoofMaterial', 'OpeningProtection', 'SecWaterResistance', 'SinkholeTerritory', 'HurricaneTerritory', 'AOPTerritory', 'WaterTerritory', 'CoverageAPremium', 'CoverageBPremium', 'CoverageCPremium', 'CoverageDPremium', 'CoverageEPremium', 'CoverageFPremium', 'SeniorDiscount', 'WindMitigationFeatures', 'PaperlessDiscount', 'WaterLeakDetectionCredit', 'RoofDeductibleCredit', 'RoofDeductibleCovA', 
    #'GolfCartDetails', 
    'noOfGolfCart', 
    #'DogDetails', 
    'NumberOfDogs', 'ProtectionClass.County', 'ProtectionClass.Protected', 'ProtectionClass.UnProtected', 'ProtectionClass.EffectiveDate', 'BCEGDetails.CommunityName', 'BCEGDetails.County', 'BCEGDetails.BCEGNumber', 'BCEGDetails.BeginYear'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 84, Finished, Available, Finished)

In [132]:
mergeandinsert('Risks_Properties',df_property1,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 85, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`


Merged new data into Tables/Risks_Properties_versions


In [133]:
mergeandinsert('Risks_Property_Premium',df_propertypremium,['Policy_ref','Property_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 86, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Property_Id` = df.`Property_Id`


Merged new data into Tables/Risks_Property_Premium_versions


In [134]:
df_propertyriskattributes

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 87, Finished, Available, Finished)

,Policy_ref,Property_Id,DTCGrouping,DwellingAge,AgeOfRoof,RCE,DescribeRoofType,RoofYear,SidingType,SalesPrice,...,noOfGolfCart,NumberOfDogs,ProtectionClass.County,ProtectionClass.Protected,ProtectionClass.UnProtected,ProtectionClass.EffectiveDate,BCEGDetails.CommunityName,BCEGDetails.County,BCEGDetails.BCEGNumber,BCEGDetails.BeginYear
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,EAST COAST,1,1,610840,,2024,Hardiplank/Hardi Panel (Cement Fiber),78555,...,0,,,2,,,,,3,2012
1,ae46254b-2338-44ed-8238-486145c927fc,1,EAST COAST,1,1,461995,,2024,Hardiplank/Hardi Panel (Cement Fiber),76576,...,0,,,3,,,,,4,2012
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,EAST COAST,1,1,343496,,2024,Hardiplank/Hardi Panel (Cement Fiber),57655,...,0,,,3,,,,,4,2012
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,SOUTH EAST,0,0,619708,,2025,Hardiplank/Hardi Panel (Cement Fiber),600000,...,0,,BROWARD,1,-,2016-08-01,Coral Springs,Broward,3,1996
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,WEST COAST,0,1,398487,,2024,Hardiplank/Hardi Panel (Cement Fiber),76776,...,0,,,2,,,,,2,2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,EAST COAST,1,1,461995,,2024,Hardiplank/Hardi Panel (Cement Fiber),322342,...,0,,Osceola,3,,,,,4,2012
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,EAST COAST,1,1,378562,,2024,Hardiplank/Hardi Panel (Cement Fiber),352325,...,0,,Saint Johns,3,,,,,3,2010
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,EAST COAST,1,1,358619,,2024,Hardiplank/Hardi Panel (Cement Fiber),453452,...,0,,Osceola,3,,,,,4,2012
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,EAST COAST,1,1,358637,,2024,Hardiplank/Hardi Panel (Cement Fiber),456777,...,0,,Volusia,2,,,,,3,2012


In [135]:
mergeandinsert('Risks_Property_PropertyRiskAttributes',df_propertyriskattributes,['Policy_ref','Property_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 88, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Property_Id` = df.`Property_Id`
Merged new data into Tables/Risks_Property_PropertyRiskAttributes_versions


In [136]:
df_propertyriskattributes

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 89, Finished, Available, Finished)

,Policy_ref,Property_Id,DTCGrouping,DwellingAge,AgeOfRoof,RCE,DescribeRoofType,RoofYear,SidingType,SalesPrice,...,noOfGolfCart,NumberOfDogs,ProtectionClass.County,ProtectionClass.Protected,ProtectionClass.UnProtected,ProtectionClass.EffectiveDate,BCEGDetails.CommunityName,BCEGDetails.County,BCEGDetails.BCEGNumber,BCEGDetails.BeginYear
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,EAST COAST,1,1,610840,,2024,Hardiplank/Hardi Panel (Cement Fiber),78555,...,0,,,2,,,,,3,2012
1,ae46254b-2338-44ed-8238-486145c927fc,1,EAST COAST,1,1,461995,,2024,Hardiplank/Hardi Panel (Cement Fiber),76576,...,0,,,3,,,,,4,2012
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,EAST COAST,1,1,343496,,2024,Hardiplank/Hardi Panel (Cement Fiber),57655,...,0,,,3,,,,,4,2012
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,SOUTH EAST,0,0,619708,,2025,Hardiplank/Hardi Panel (Cement Fiber),600000,...,0,,BROWARD,1,-,2016-08-01,Coral Springs,Broward,3,1996
4,06748658-94cb-49fa-83e5-6a36813b41fd,1,WEST COAST,0,1,398487,,2024,Hardiplank/Hardi Panel (Cement Fiber),76776,...,0,,,2,,,,,2,2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,1,EAST COAST,1,1,461995,,2024,Hardiplank/Hardi Panel (Cement Fiber),322342,...,0,,Osceola,3,,,,,4,2012
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,EAST COAST,1,1,378562,,2024,Hardiplank/Hardi Panel (Cement Fiber),352325,...,0,,Saint Johns,3,,,,,3,2010
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,EAST COAST,1,1,358619,,2024,Hardiplank/Hardi Panel (Cement Fiber),453452,...,0,,Osceola,3,,,,,4,2012
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,EAST COAST,1,1,358637,,2024,Hardiplank/Hardi Panel (Cement Fiber),456777,...,0,,Volusia,2,,,,,3,2012


In [137]:
df_propertyriskattributes_dogdetails

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 90, Finished, Available, Finished)

,Policy_ref,DD_id,SerialNo,DogBreed,PastHistoryOfBiteOrAttack
0,17ad30ed-473e-4533-a84e-280606638754,1,1,pug,No
1,9199b761-388b-41eb-832b-133035df5dc6,1,1,pug,No
2,dadc07ac-18cb-47b5-b25d-2eb2b4069443,1,1,pug,No


In [138]:
mergeandinsert('Risks_Property_PropertyRiskAttributes_DogDetails',df_propertyriskattributes_dogdetails,['Policy_ref','DD_id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 91, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`DD_id` = df.`DD_id`
Merged new data into Tables/Risks_Property_PropertyRiskAttributes_DogDetails_versions


In [139]:
mergeandinsert('Risks_Property_PropertyRiskAttributes_GolfCartDetails',df_propertyriskattributes_golf,['Policy_ref','GCD_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 92, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`GCD_Id` = df.`GCD_Id`


Merged new data into Tables/Risks_Property_PropertyRiskAttributes_GolfCartDetails_versions


In [140]:
mergeandinsert('Risks_Property_Address',df_propertyaddress,['Policy_ref','Property_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 93, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Property_Id` = df.`Property_Id`


Merged new data into Tables/Risks_Property_Address_versions


In [141]:
mergeandinsert('Risks_Property_Coverages',df_propertycoverages,['Policy_ref','Property_Id','Name'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 94, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Property_Id` = df.`Property_Id` AND old_data.`Name` = df.`Name`


Merged new data into Tables/Risks_Property_Coverages_versions


In [142]:
mergeandinsert('Risks_Property_PremiumFactors',df_propertypremiumfactors,['Policy_ref','Property_Id','Name'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 95, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Property_Id` = df.`Property_Id` AND old_data.`Name` = df.`Name`


Merged new data into Tables/Risks_Property_PremiumFactors_versions


## TransactionHistory

In [143]:
df_main = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    trans_his = 0
    if jsonn['TransactionHistory']:
        for json_data in jsonn['TransactionHistory']:
            trans_his += 1
            json_data = {'Policy_ref': ref, 'TransactionHistory_Id': trans_his, **json_data}
            df_main.append(json_data)

df_main = pd.json_normalize(df_main)
df_main.replace({np.nan: None}, inplace=True)
df_main


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 96, Finished, Available, Finished)

,Policy_ref,TransactionHistory_Id,Type,Date,EffectiveDate,Number,Remarks,EffectivePremium,AnnualPremium,EffectivePremiumWithFeesAndTaxes,AnnualPremiumWithFeesAndTaxes,UpdatedBy,UpdatedOn,AnnualFees,AnnualTax,Fees,Taxes
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1,Policy,2025-09-22,2025-09-22,0,,2064.0,2064,2069.86,2069.86,Abigail Miller,2025-09-22T13:38:27.702Z,5.86,0,5.86,0
1,ae46254b-2338-44ed-8238-486145c927fc,1,Policy,2025-09-24,2025-09-24,0,,2999.0,2999,2995.51,2995.51,Underwriter User,2025-09-24T07:56:46.58Z,-3.49,0,-3.49,0
2,794deccd-0359-4ca3-b72f-a9d9ead11391,1,Policy,2025-09-24,2025-09-24,0,,3745.0,3745,3734.05,3734.05,Underwriter User,2025-09-24T12:54:31.238Z,-10.95,0,-10.95,0
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,1,Policy,2025-07-09,2025-07-09,0,,8291.0,8291,8234.59,8234.59,Underwriter User,2025-07-09T14:49:23.613Z,-56.41,0,-56.41,0
4,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,2,Cancellation,2025-09-24,2025-07-09,1,,-8291.0,8291,-8234.59,8234.59,Underwriter User,2025-09-07T22:01:25.256Z,-56.41,0,56.41,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,1,Policy,2025-10-15,2025-09-15,0,,970.0,970,986.79,986.79,Underwriter User,2025-10-15T12:48:23.546Z,16.79,0,16.79,0
202,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,1,Policy,2025-10-15,2025-10-15,0,,2570.0,2570,2622.70,2622.70,Underwriter User,2025-10-15T12:49:15.727Z,52.70,0,52.70,0
203,31e77022-e1fa-4725-a7b1-d8c2fd500b85,1,Policy,2025-10-15,2025-10-29,0,,982.0,982,1018.82,1018.82,Underwriter User,2025-10-15T12:52:28.544Z,36.82,0,36.82,0
204,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,1,Policy,2025-10-15,2025-10-29,0,,982.0,982,1018.82,1018.82,Underwriter User,2025-10-15T12:52:28.544Z,36.82,0,36.82,0


In [144]:
mergeandinsert('TransactionHistory',df_main,['Policy_ref','Number'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 97, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Number` = df.`Number`


Merged new data into Tables/TransactionHistory_versions


### TransactionFeesAndTaxes

In [14]:
# # Check duplicates based on 'Policy_ref' and 'Id'
# duplicates = df.groupby(['Policy_ref', 'Id']).size().reset_index(name='count')
# duplicates = duplicates[duplicates['count'] > 1]

# print(duplicates)


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 98, Finished, Available, Finished)

In [ ]:
# df = spark.sql("SELECT 'Policy_ref','TransactionFeesAndTaxes_Id' FROM UAT_DIEP2_EMBARK.TransactionFeesAndTaxes_versions ")
# display(df)

In [11]:
df = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    if 'TransactionFeesAndTaxes' in jsonn and jsonn['TransactionFeesAndTaxes']:
        trans_id = 0
        for json_data in jsonn['TransactionFeesAndTaxes']:
            trans_id += 1
            if 'ProductFeesAndTaxes' in json_data and isinstance(json_data['ProductFeesAndTaxes'], dict):
                json_data['ProductFeesAndTaxes'].setdefault('Status', "")
            json_data = {'Policy_ref': ref, 'TransactionFeesAndTaxes_Id': trans_id, **json_data}
            df.append(json_data)

df = pd.json_normalize(df)
if 'ProductFeesAndTaxes.Status' in df.columns:
    df['ProductFeesAndTaxes.Status'] = df['ProductFeesAndTaxes.Status'].apply(lambda x: "" if pd.isna(x) else x)
df.replace({np.nan: None}, inplace=True)
df


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 99, Finished, Available, Finished)

,Policy_ref,Id,Code,Description,Type,IsOverride,Status,Value,ValueType,Amount,AnnualAmount,ProductFeesAndTaxes,Name,CreatedOn,CreatedBy,UpdatedOn,UpdatedBy
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,1.0,FASTFEE,2023 FIGA Assessment Fee,Fee,False,Active,0.01,P,20.64,20.64,None,None,None,None,None,None
1,111b5c8b-30b3-427f-977b-fe5ccd358b5f,2.0,MGAFEE,MGA Fee,Fee,False,Active,25.0,V,25.0,25.0,None,None,None,None,None,None
2,111b5c8b-30b3-427f-977b-fe5ccd358b5f,3.0,EMPATRF,EMPA Trust Fund,Fee,False,Active,2.0,V,2.0,2.0,None,None,None,None,None,None
3,111b5c8b-30b3-427f-977b-fe5ccd358b5f,4.0,SETUPFEE,Setup Fee,Fee,False,Active,0.0,V,0.0,0.0,None,None,None,None,None,None
4,111b5c8b-30b3-427f-977b-fe5ccd358b5f,5.0,LFMAD,LFMA Discount,Fee,False,Active,-5.16,V,-5.16,-5.16,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,4.0,LPTD,LPT Discount,Fee,False,Active,0.0,V,0.0,0.0,None,None,None,None,None,None
756,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,5.0,LPTD-MGA,LPT Discount - MGA,Fee,False,InActive,0.0,V,0.0,0.0,None,None,None,None,None,None
757,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,6.0,LFMAD,LFMA Discount,Fee,False,Active,0.0,V,0.0,0.0,None,None,None,None,None,None
758,6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc,7.0,LFMADMGA,LFMA Discount - MGA,Fee,False,InActive,0.0,V,0.0,0.0,None,None,None,None,None,None


In [12]:
df.columns

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 100, Finished, Available, Finished)

Index(['Policy_ref', 'Id', 'Code', 'Description', 'Type', 'IsOverride',
       'Status', 'Value', 'ValueType', 'Amount', 'AnnualAmount',
       'ProductFeesAndTaxes', 'Name', 'CreatedOn', 'CreatedBy', 'UpdatedOn',
       'UpdatedBy'],
      dtype='object')

In [14]:
# # Step 1: Identify boolean columns by dtype
# bool_cols = df.select_dtypes(include=['bool', 'object']).columns.tolist()

# # Step 2: For each boolean column, find rows where values are not True, False, or None
# for col_name in bool_cols:
#     invalid_rows = df[~df[col_name].isin([True, False, None])]
#     if not invalid_rows.empty:
#         print(f"Invalid values found in column '{col_name}':")
#         print(invalid_rows[['Policy_ref', col_name]])


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 101, Finished, Available, Finished)

In [1]:
# print(df['ProductFeesAndTaxes.Status'])


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 102, Finished, Available, Finished)

In [11]:
mergeandinsert('TransactionFeesAndTaxes',df,['Policy_ref','TransactionFeesAndTaxes_Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 103, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Id` = df.`Id`


Merged new data into Tables/TransactionFeesAndTaxes_versions


In [ ]:
# Issue identification
# print(df.dtypes)


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 104, Finished, Available, Finished)

In [ ]:
# for col in df.columns:
#     if df[col].astype(str).str.contains("FASTFEE").any():
#         print(f"⚠️ Found 'FASTFEE' in column: {col}")


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 105, Finished, Available, Finished)

In [ ]:
# for col in df.columns:
#     mask = df[col].astype(str).str.contains("FASTFEE", na=False)
#     if mask.any():
#         print(f"⚠️ Found 'FASTFEE' in column: {col}")
#         print(df.loc[mask, ["Policy_ref", col]])


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 106, Finished, Available, Finished)

### Rules

In [ ]:
df = []
df_m = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    if jsonn['Rules']:
        json_data = jsonn['Rules']
        mr_id = 0
        if json_data['MatchingRules']:
            for d in json_data['MatchingRules']:
                mr_id += 1
                d = {'Policy_ref': ref, 'Id': mr_id, **d}
                df_m.append(d)
        json_data = {'Policy_ref': ref, **json_data}
        df.append(json_data)

df = pd.json_normalize(df)
df_m = pd.json_normalize(df_m)

df.replace({np.nan: None}, inplace=True)
df_m.replace({np.nan: None}, inplace=True)

df, df_m


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 107, Finished, Available, Finished)

(                              Policy_ref     Action MatchingRules
 0   111b5c8b-30b3-427f-977b-fe5ccd358b5f      Allow            []
 1   ae46254b-2338-44ed-8238-486145c927fc      Allow            []
 2   794deccd-0359-4ca3-b72f-a9d9ead11391      Allow            []
 3   0410bcb2-f2c6-48c3-b8b1-99101ce774d7      Allow            []
 4   06748658-94cb-49fa-83e5-6a36813b41fd      Allow            []
 ..                                   ...        ...           ...
 90  c831e867-b099-4812-909a-febb8c464d40      Allow            []
 91  1f6893f2-faa5-4cfc-a2fa-8689a3b838f3      Allow            []
 92  aa5f4d7d-6b0a-4631-bda7-52c98c8c072d      Allow            []
 93  31e77022-e1fa-4725-a7b1-d8c2fd500b85      Allow            []
 94  6ea0a5ea-4fd2-4e9a-b617-69734c0f5fcc  ReferToUW            []
 
 [95 rows x 3 columns],
 Empty DataFrame
 Columns: []
 Index: [])

In [ ]:
df=df.iloc[:,:-1]
df

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 108, Finished, Available, Finished)

,Policy_ref,Action
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,Allow
1,ae46254b-2338-44ed-8238-486145c927fc,Allow
2,794deccd-0359-4ca3-b72f-a9d9ead11391,Allow
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,Allow
4,06748658-94cb-49fa-83e5-6a36813b41fd,Allow
...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,Allow
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,Allow
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,Allow
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,Allow


In [ ]:
mergeandinsert('Rules',df,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 109, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/Rules_versions


In [ ]:
df_m

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 110, Finished, Available, Finished)

""


In [ ]:
mergeandinsert('Rules_MatchingRules',df_m,['Policy_ref','Type','Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 111, Finished, Available, Finished)

Skip since empty


### DisplayRules

In [ ]:
df = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    trans_id = 0
    if jsonn['DisplayRules']:
        for json_data in jsonn['DisplayRules']:
            trans_id += 1
            json_data = {'Policy_ref': ref, 'Id': trans_id, **json_data}
            df.append(json_data)

df = pd.json_normalize(df)
df.replace({np.nan: None}, inplace=True)
df


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 112, Finished, Available, Finished)

,Policy_ref,Id,Reason,Header
0,33a8fd03-af6b-432e-b49d-c66267f10919,1,Coverage A in $300K–$350K range.,New Buisness
1,200a6c2e-5a0c-4332-8f4c-ce602e60628e,1,Coverage A in $300K–$350K range.,New Buisness
2,17ad30ed-473e-4533-a84e-280606638754,1,Limited or Excluded selected for Water Damage ...,New Buisness
3,9199b761-388b-41eb-832b-133035df5dc6,1,Policyholder's First Name is updated,1st Endorsement
4,f8ba8a5d-e043-4fb7-8f2f-b25aa05f3bbd,1,Limited or Excluded selected for Water Damage ...,New Buisness
5,dadc07ac-18cb-47b5-b25d-2eb2b4069443,1,Limited or Excluded selected for Water Damage ...,1st Endorsement
6,e6deecf6-fb72-4b6c-9aff-35120b4cb929,1,Policyholder's First Name is updated,1st Endorsement
7,e782a22d-3853-4ba7-8159-5af0c659ac71,1,Coins - Scheduled Personal Property Individual...,1st Endorsement
8,8ade4103-f685-44f4-8f6c-c76d2ac8d91e,1,Coverage A in $300K–$350K range.,New Buisness
9,47180e80-53b6-479d-8e18-cbe5316acc74,1,Coverage A in $300K–$350K range.,New Buisness


In [ ]:
mergeandinsert('DisplayRules',df,['Policy_ref','Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 113, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Id` = df.`Id`
Merged new data into Tables/DisplayRules_versions


### PolicyLog

In [ ]:
df = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    trans_id = 0
    if jsonn['PolicyLog']:
        for json_data in jsonn['PolicyLog']:
            trans_id += 1
            json_data = {'Policy_ref': ref, 'Id': trans_id, **json_data}
            df.append(json_data)

df = pd.json_normalize(df)
df.replace({np.nan: None}, inplace=True)
df


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 114, Finished, Available, Finished)

,Policy_ref,Id,fieldName,oldValue,newValue,Status,Header
0,f0377d71-5b55-4f30-a62e-c8e5649dbed1,1,Other Structures (Coverage B),2% of Cov A,10% of Cov A,Edited,Coverage & Limit Options
1,1c6c58ec-72c3-4a31-b1da-fb863d2edcdf,1,Insured Mobile Number,987-123-1233,987-123-1244,Edited,Policy Holder Information
2,035e8fdb-2d47-478f-9165-81c2be3b79d9,1,Insured Mobile Number,987-123-1233,987-123-1244,Edited,Policy Holder Information
3,5612b267-a53e-4192-aac5-6d79e5e59468,1,Insured Mobile Number,987-123-1233,987-123-1244,Edited,Policy Holder Information
4,17ad30ed-473e-4533-a84e-280606638754,1,Increased Replacement Cost on Dwelling,No,Yes,Edited,Coverage & Limit Options
5,17ad30ed-473e-4533-a84e-280606638754,2,Computer Equipment Coverage,"$7,000","$15,000",Edited,Coverage & Limit Options
6,7cb48e62-7e24-4c6d-bfb0-9bcd3d8692c6,1,1st Additional Insured Name,--,test_add_insured,Added,Additional Insured
7,7cb48e62-7e24-4c6d-bfb0-9bcd3d8692c6,2,1st Additional Insured Address,--,"test, test, TE 11111",Added,Additional Insured
8,7cb48e62-7e24-4c6d-bfb0-9bcd3d8692c6,3,1st Additional Insured Explain the Insurable I...,--,test2311,Added,Additional Insured
9,7cb48e62-7e24-4c6d-bfb0-9bcd3d8692c6,4,Screen Enclosure and/or Carport Limit,"$25,000","$20,000",Edited,Coverage & Limit Options


In [ ]:
mergeandinsert('PolicyLog',df,['Policy_ref','Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 115, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref` AND old_data.`Id` = df.`Id`
Merged new data into Tables/PolicyLog_versions


### PaymentTerms

In [ ]:
df = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    if jsonn['PaymentTerms']:
        json_data = jsonn['PaymentTerms']
        json_data = {'Policy_ref': ref, **json_data}
        df.append(json_data)

df = pd.json_normalize(df)
df.replace({np.nan: None}, inplace=True)
df


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 116, Finished, Available, Finished)

,Policy_ref,DownPayment,FirstDueDate,PaymentAmount,Installments,PayOption,PayPlan,OutsidePF,PaidAmount,NeedIPFS,OutsidePFDownPayment,AdjustedDownPayment
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,0,,0,0,Credit Card,Quarterly,,0,True,0,0
1,ae46254b-2338-44ed-8238-486145c927fc,0,,0,0,Mortgagee Bill,Escrow,,0,True,0,0
2,794deccd-0359-4ca3-b72f-a9d9ead11391,0,,0,0,Credit Card,Quarterly,,0,True,0,0
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,0,,0,0,Mortgagee Bill,Escrow,,0,True,0,0
4,06748658-94cb-49fa-83e5-6a36813b41fd,0,,0,0,Credit Card,Monthly,,0,True,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,0,,0,0,Credit Card,Monthly,,0,True,0,0
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,0,,0,0,Credit Card,Monthly,,0,True,0,0
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,0,,0,0,Mortgagee Bill,Escrow,,0,True,0,0
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,0,,0,0,Mortgagee Bill,Escrow,,0,True,0,0


In [ ]:
mergeandinsert('PaymentTerms',df,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 117, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/PaymentTerms_versions


### ExtraEmails

In [ ]:
df = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    trans_id = 0
    if jsonn['ExtraEmails']:
        for json_data in jsonn['ExtraEmails']:
            trans_id += 1
            json_data = {'Policy_ref': ref, 'Id': trans_id, 'Email': json_data}
            df.append(json_data)

df = pd.json_normalize(df)
df.replace({np.nan: None}, inplace=True)
df


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 118, Finished, Available, Finished)

""


In [ ]:
mergeandinsert('ExtraEmails',df,['Policy_ref','Id'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 119, Finished, Available, Finished)

Skip since empty


### BillingAttributes

In [ ]:
df = []
for jsonn in landing_data:
    ref = policy_ref(jsonn)
    if 'BillingAttributes' in jsonn and jsonn['BillingAttributes']:
        json_data = jsonn['BillingAttributes']
        json_data = {'Policy_ref': ref, **json_data}
        df.append(json_data)

df = pd.json_normalize(df)
df.replace({np.nan: None}, inplace=True)
df


StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 120, Finished, Available, Finished)

,Policy_ref,BillingAccountNumber,InstallmentDueDate,EstimatedBalanceAfterCancel,CurrentPolicyBalance,CurrentInstallmentAmount
0,111b5c8b-30b3-427f-977b-fe5ccd358b5f,100001032,,0.0,0.00,None
1,ae46254b-2338-44ed-8238-486145c927fc,100001069,,0.0,0.00,None
2,794deccd-0359-4ca3-b72f-a9d9ead11391,100001070,,0.0,0.00,None
3,0410bcb2-f2c6-48c3-b8b1-99101ce774d7,100000992,None,0.0,8234.59,None
4,06748658-94cb-49fa-83e5-6a36813b41fd,100001071,,0.0,0.00,None
...,...,...,...,...,...,...
90,c831e867-b099-4812-909a-febb8c464d40,100001104,,0.0,0.00,None
91,1f6893f2-faa5-4cfc-a2fa-8689a3b838f3,100001105,,0.0,0.00,None
92,aa5f4d7d-6b0a-4631-bda7-52c98c8c072d,100001106,,0.0,0.00,None
93,31e77022-e1fa-4725-a7b1-d8c2fd500b85,100001107,,0.0,0.00,None


In [ ]:
mergeandinsert('BillingAttributes',df,['Policy_ref'])

StatementMeta(, 7fd2f6e9-1879-49fb-b814-0d23ea853b47, 121, Finished, Available, Finished)

Merge_condition =  old_data.`Policy_ref` = df.`Policy_ref`
Merged new data into Tables/BillingAttributes_versions
